In [2]:
# Import libraries
import pandas as pd
import time
import psutil
import os
import logging
import re
from pathlib import Path
from datetime import datetime
import gc
import threading

In [3]:
# Import other libraries
from typing import Any, Literal

#### Setup Logger

In [4]:
# Initialises and returns a configured logger
def setup_logger(
    log_path: str | Path = "logs/pipeline.log"
) -> logging.Logger:
    # Ensure the log directory exists
    Path(log_path).parent.mkdir(parents = True, exist_ok = True)

    logger = logging.getLogger("pipeline_logger")
    logger.setLevel(logging.INFO)

    if not logger.handlers:
        file_handler = logging.FileHandler(log_path, mode = "a", encoding = "utf-8")
        formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)

    return logger

In [5]:
# Logs an informational message using the provided logger
def log_info_msg(
    logger: logging.Logger, 
    msg: str
) -> None:
    logger.info(msg)

In [6]:
# Logs an error message using the provided logger
def log_error_msg(
    logger: logging.Logger, 
    msg: str
) -> None:
    logger.error(msg)

In [7]:
# Create logger
logger = setup_logger()

#### Resource Monitore class and Timer class

In [8]:
# Background monitor class for RSS (memory), CPU usage, per core load and I/O throughput
class ResourceMonitor:

    # Initialise monitor class
    def __init__(
        self, 
        interval: float = 0.1, 
        include_children: bool = True, 
        scale: int = 1024 * 1024
    ):
        self.interval = interval
        self.include_children = include_children
        self.process = psutil.Process(os.getpid())
        self._stop_monitoring = threading.Event()
        self._thread = None
        self._io_initial_rw = None
        self._scale = scale

        self.samples = {
            "rss_mib": [],
            "process_cpu_percent": [],
            "system_cpu_per_core_percent": [],
            "read_mib": [],
            "write_mib": []
        }

    # Return child processes of the monitored process
    def _get_children(
        self
    ) -> list[psutil.Process]:
        try:
            return self.process.children(recursive = True)
        except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess) as e:
            print(f"Failed get children: {type(e).__name__}: {e}")
        return []

    # Apply a metric function to process + children and return the aggregated value
    def _sum_process_metric(
        self, 
        function: callable
    ) -> float:
        total = 0
        procs = [self.process]
        if self.include_children:
            procs.extend(self._get_children())

        for p in procs:
            try:
                total += function(p)
            except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess) as e:
                print(f"Failed to add metric: {type(e).__name__}: {e}")
        return total

    # Return current RSS memory usage in MiB
    def _get_rss_mib(self) -> float:
        rss_bytes = self._sum_process_metric(lambda p: p.memory_info().rss)
        return rss_bytes / self._scale 

    # Return aggregated read/write I/O in MiB
    def _get_io_mib(self) -> tuple[float, float]:

        def read_bytes(p):
            return p.io_counters().read_bytes

        def write_bytes(p):
            return p.io_counters().write_bytes

        read_b = self._sum_process_metric(read_bytes)
        write_b = self._sum_process_metric(write_bytes)
        return read_b / self._scale, write_b / self._scale

    # Initialise CPU counters
    def _initialise_cpu_utilisation(self) -> None:
        try:
            self.process.cpu_percent(interval = None)
        except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess) as e:
            print(f"Failed to return metric (cpu utilisation) per cpu: {type(e).__name__}: {e}")

        if self.include_children:
            for p in self._get_children():
                try:
                    p.cpu_percent(interval = None)
                except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess) as e:
                    print(f"Failed to return cpu metric (utilisation) of the children processes: {type(e).__name__}: {e}")

        try:
            psutil.cpu_percent(interval = None, percpu = True)
        except Exception as e:
            print(f"Failed to return cpu metric (utilisation) per cpu: {type(e).__name__}: {e}")

    # Return CPU utilization percent for process + children
    def _get_process_cpu_utilisation_percent(self) -> float:
        return self._sum_process_metric(lambda p: p.cpu_percent(interval = None))

    # Background loop to collect metrics
    def _run(self) -> None:
        self._initialise_cpu_utilisation()
        read_initial, write_initial = self._get_io_mib()
        self._io_initial_rw = (read_initial, write_initial)

        while not self._stop_monitoring.is_set():
            rss_mib = self._get_rss_mib()
            process_cpu_utilisation = self._get_process_cpu_utilisation_percent()
            
            try:
                system_per_core_utilisation = psutil.cpu_percent(interval = None, percpu = True)
            except Exception as e:
                print(f"Failed to return metric (cpu utilisation) per cpu: {type(e).__name__}: {e}")
                system_per_core_utilisation = []
    
            read_mib, write_mib = self._get_io_mib()

            self.samples["rss_mib"].append(rss_mib)
            self.samples["process_cpu_percent"].append(process_cpu_utilisation)
            if len(system_per_core_utilisation) > 0:
                self.samples["system_cpu_per_core_percent"].append(system_per_core_utilisation)
            self.samples["read_mib"].append(read_mib - self._io_initial_rw[0])
            self.samples["write_mib"].append(write_mib - self._io_initial_rw[1])

            time.sleep(self.interval)

    # Start background resource monitoring
    def start(self) -> None:
        self._stop_monitoring.clear()
        self._thread = threading.Thread(target = self._run, daemon = True)
        self._thread.start()


    # Stop monitoring and return aggregated statistics
    def stop(self) -> dict[str, float | list[float]]:
        self._stop_monitoring.set()
        if self._thread is not None:
            self._thread.join()

        def avg(l):
            return sum(l) / len(l) if l else 0.0

        def peak(l):
            return max(l) if l else 0.0

        per_core_samples = self.samples["system_cpu_per_core_percent"]
        ncores = len(per_core_samples[0]) if per_core_samples else psutil.cpu_count(logical = True) or 1

        avg_per_core = [0.0] * ncores
        peak_per_core = [0.0] * ncores

        if per_core_samples:
            for core_idx in range(ncores):
                vals = [sample[core_idx] for sample in per_core_samples]
                avg_per_core[core_idx] = sum(vals) / len(vals)
                peak_per_core[core_idx] = max(vals)
                
        peak_rss_mib = peak(self.samples["rss_mib"])
        total_ram_mib = psutil.virtual_memory().total / self._scale
        memory_percent_of_total = 100.0 * peak_rss_mib / total_ram_mib if total_ram_mib > 0 else 0.0

        return {
            "avg_rss_mib": avg(self.samples["rss_mib"]),
            "peak_rss_mib": peak_rss_mib,
            "memory_percent_of_total": memory_percent_of_total,
            "avg_process_cpu_percent": avg(self.samples["process_cpu_percent"]),
            "peak_process_cpu_percent": peak(self.samples["process_cpu_percent"]),
            "avg_system_cpu_per_core_percent": avg_per_core,
            "peak_system_cpu_per_core_percent": peak_per_core,
            "read_mib": peak(self.samples["read_mib"]),
            "write_mib": peak(self.samples["write_mib"])
        }

    # Static method that return an empty/default resource statistics dictionary
    @staticmethod
    def as_dict() -> dict[str, float | list[float]]:
        return {
            "avg_rss_mib": 0.0,
            "peak_rss_mib": 0.0,
            "memory_percent_of_total": 0.0,
            "avg_process_cpu_percent": 0.0,
            "peak_process_cpu_percent": 0.0,
            "avg_system_cpu_per_core_percent": [],
            "peak_system_cpu_per_core_percent": [],
            "read_mib": 0.0,
            "write_mib": 0.0 
        }
# Timer class for benchmarking source code sections
class Timer:
    # Initialise timer class
    def __init__(self):
        self.times = {}

    # Start a named timer
    def start(self, name: str) -> None:
        self.times[f"{name}_start"] = time.perf_counter()

    # Stop a named timer
    def stop(self, name: str) -> None:
        self.times[f"{name}_end"] = time.perf_counter()

    # Return execution time for a named timer
    def duration(self, name: str) -> float:
        return self.times[f"{name}_end"] - self.times[f"{name}_start"]

    # Return current timestamp
    @staticmethod
    def get_timestamp() -> datetime:
        return datetime.now().astimezone()


In [9]:
# Compute the dataset_size
def compute_dataset_size_mib(
    path: str | Path
) -> float:
    path = Path(path)
    if not path.exists():
        return 0.0
    total_bytes = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
    return total_bytes / (1024 * 1024)

#### Sample-based estimator 

##### Algorithm to determine if given CSV files (a slice) could be processed entirely in memory or chunked ingestion is needed instead of 

In [10]:
# Constant variables
CSV_PATTERN = "*.csv"
IN_MEMORY = "in-memory"
CHUNKING = "chunking"
STANDARD = "standard"

SLICE_PATHS = [
    "../data/CSV/raw-data/1st-slice/",
    "../data/CSV/raw-data/2nd-slice/",
    "../data/CSV/raw-data/3rd-slice/",
    "../data/CSV/raw-data/4th-slice/",
]
INGESTION = "ingestion"
CLEANING_TRANSFORMATION = "ingestion, cleaning, transformation and profiling"
PERSISTENCE_TO_PARQUET = "ingestion, cleaning, transformation, profiling and persistence to parquet"
COMPACTION_TO_PARQUET_FILE = "compaction to one parquet file per partition"
QUERIES = "queries"
MACHINE_LEARNING = "machine learning"

In [11]:
# Return the number of data rows in a CSV file
def count_rows_in_csv(
    file_path: str
) -> int:
    with open(file_path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f) - 1

In [12]:
# Return total number of rows across all CSV files in a slice 
def count_rows_in_slice(
    slice_path: str
) -> int:
    files = sorted(glob.glob(os.path.join(slice_path, CSV_PATTERN)))
    return sum(count_rows_in_csv(f) for f in files)

In [13]:
# Estimate the memory needed to load a slice of data and decide between in-memory or chunking mode
def estimate_slice_memory_with_sample(
    slice_path: str, 
    sample_rows: int = 100_000, 
    safe_ratio: float = 0.6
) -> dict[str, float | int | str] :
    files = sorted(glob.glob(os.path.join(slice_path, CSV_PATTERN)))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {slice_path}")

    # Count total rows in the slice
    total_rows = count_rows_in_slice(slice_path)

    # Read a sample from the first file
    sample_df = pd.read_csv(
        files[0],
        nrows = sample_rows,
        dtype = {"store_and_fwd_flag": "string"}, 
        low_memory = False
    )

    # Estimate bytes per row in memory
    sample_mem_bytes = sample_df.memory_usage(deep=True).sum()
    bytes_per_row = sample_mem_bytes / len(sample_df)

    # Estimate total memory needed
    estimated_memory_bytes = bytes_per_row * total_rows

    # Compare against available RAM
    available_ram_bytes = psutil.virtual_memory().available
    threshold_bytes = available_ram_bytes * safe_ratio

    mode = IN_MEMORY if estimated_memory_bytes < threshold_bytes else CHUNKING

    return {
        "slice_path": slice_path, 
        "mode": mode,
        "total_rows": int(total_rows),
        "bytes_per_row": float(bytes_per_row),
        "estimated_memory_gb": float(estimated_memory_bytes / (1024**3)),
        "available_ram_gb": float(available_ram_bytes / (1024**3)),
        "safe_threshold_gb": float(threshold_bytes / (1024**3)),
    }
    

In [159]:
# Estimated memory occupied by the slices
results = {}
slice_number = 1

for path in SLICE_PATHS:
    print(f"\n--- Processing {path} ---")
    results[f"slice_{slice_number}"] = estimate_slice_memory_with_sample(path)
    slice_number += 1

df_summary = pd.DataFrame.from_dict(results, orient='index')
print(df_summary)


--- Processing ../data/CSV/raw-data/1st-slice/ ---

--- Processing ../data/CSV/raw-data/2nd-slice/ ---

--- Processing ../data/CSV/raw-data/3rd-slice/ ---

--- Processing ../data/CSV/raw-data/4th-slice/ ---
                              slice_path       mode  total_rows  \
slice_1  ../data/CSV/raw-data/1st-slice/  in-memory     3898963   
slice_2  ../data/CSV/raw-data/2nd-slice/   chunking    12813768   
slice_3  ../data/CSV/raw-data/3rd-slice/   chunking    52359167   
slice_4  ../data/CSV/raw-data/4th-slice/   chunking   107462293   

         bytes_per_row  estimated_memory_gb  available_ram_gb  \
slice_1      346.00128             1.256397          6.124390   
slice_2      346.00128             4.129093          6.128101   
slice_3      338.00128            16.482049          6.137779   
slice_4      338.00128            33.827864          6.115513   

         safe_threshold_gb  
slice_1           3.674634  
slice_2           3.676861  
slice_3           3.682668  
slice_4       

#### Ingestion Stage

In [14]:
# Build benchmark return for ingestion step
def build_return_dict_ingestion(
    slice_path: str | Path,
    operation: str,
    ntest: int,
    processed_files: int,
    processed_rows: int,
    execution_time: float,
    throughput: float,
    benchmark_start_timestamp: str,
    benchmark_end_timestamp: str,
    msg: str,
    chunksize: int = 0,
    resource_stats: dict[str, Any] | None = None,
) -> dict[str, Any]:
    
    if resource_stats is None:
        resource_stats = ResourceMonitor.as_dict()

    return {
        # operations:
        "stage": INGESTION,
        # Type of operation
        "operation": operation,
        # Test Number
        "test_number": ntest,        
        # Slice path
        "slice_path": str(slice_path),
        # Performance
        "processed_files": processed_files,
        "processed_rows": processed_rows,
        "execution_time_sec": execution_time,
        "throughput_rows_sec": throughput,
        "chunksize": chunksize,
        "dataset_size_mib": compute_dataset_size_mib(slice_path),
        # Timestamps
        "benchmark_start_timestamp": benchmark_start_timestamp,
        "benchmark_end_timestamp": benchmark_end_timestamp,
        # Resources
        "avg_rss_mib": resource_stats["avg_rss_mib"],
        "peak_rss_mib": resource_stats["peak_rss_mib"],
        "memory_percent_of_total": resource_stats["memory_percent_of_total"],
        "avg_process_cpu_percent": resource_stats["avg_process_cpu_percent"],
        "peak_process_cpu_percent": resource_stats["peak_process_cpu_percent"],
        "avg_system_cpu_per_core_percent": resource_stats["avg_system_cpu_per_core_percent"],
        "peak_system_cpu_per_core_percent": resource_stats["peak_system_cpu_per_core_percent"],
        "read_mib": resource_stats["read_mib"],
        "write_mib": resource_stats["write_mib"],  
        # Message
        "msg": msg
    }

##### Benchmark ingestion, all the data in memory

In [15]:
# Benchmark CSV ingestion using Pandas library by iterating through all 
# CSV files in the given slice directory, counting rows, measuring execution time,
# throughput and other metrics. Only reading data, it is saved in memory.
# Formal parameters:
# slice_path: path to the directory containing CSV files for the slice
# logger: Logger instance for reporting progress and errors.
# ntest: test number
# Returns:
# benchmark results including slice path, processed files, processed rows, execution time, throughput, timestamps, 
# resource statistics and a message
def benchmark_pandas_ingestion_read_only(
    slice_path: str | Path, 
    ntest: int,
    logger: logging.Logger | None = None
) -> dict[str, Any]:

    if logger is None:
        logger = setup_logger()

    timer = Timer()
    monitor = ResourceMonitor()
    benchmark_start_timestamp = Timer.get_timestamp()
    timer.start("total")
    monitor.start()
    MSG_ERROR = "Ingestion finished with errors, check pipeline.log"
    
    slice_path = Path(slice_path)
    files = sorted(slice_path.glob(CSV_PATTERN))

    if len(files) == 0:
        log_error_msg(logger, f"No CSV files found in folder: {slice_path}")
        timer.stop("total")
        benchmark_end_timestamp = Timer.get_timestamp()
        resource_stats = monitor.stop()
        execution_time = timer.duration("total")
        return build_return_dict_ingestion(slice_path = slice_path, operation = IN_MEMORY, ntest = ntest, processed_files = 0, processed_rows = 0, 
                                           execution_time = execution_time, throughput = 0.0, benchmark_start_timestamp =  benchmark_start_timestamp.isoformat(),
                                           benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), resource_stats = resource_stats,
                                           msg = MSG_ERROR)
        
    log_info_msg(logger, "NYC Yellow trip taxi dataset")
    processed_files = 0
    processed_rows = 0

    for f in files:
        
        print(f"Reading {f}...")

        try:
            
            df_aux = pd.read_csv(f, low_memory = False)
            processed_rows += len(df_aux)

        except Exception as e:
            log_error_msg(logger, f"Failed to open file {f}: {type(e).__name__}: {e}")
            timer.stop("total")
            benchmark_end_timestamp = Timer.get_timestamp()
            resource_stats = monitor.stop()
            execution_time = timer.duration("total")
            throughput = processed_rows / execution_time if execution_time > 0 else 0
            return build_return_dict_ingestion(slice_path = slice_path, operation = IN_MEMORY, ntest = ntest, processed_files = processed_files, processed_rows = processed_rows, 
                                               execution_time = execution_time, throughput = throughput, benchmark_start_timestamp =  benchmark_start_timestamp.isoformat(),
                                               benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), resource_stats = resource_stats,
                                               msg = MSG_ERROR)
            
        del df_aux
        gc.collect()
        processed_files += 1
    
    timer.stop("total")
    benchmark_end_timestamp = Timer.get_timestamp()
    resource_stats = monitor.stop()
    

    execution_time = timer.duration("total")
    throughput = processed_rows / execution_time if execution_time > 0 else 0.0
    
    print(f"Pandas ingested {processed_files} file(s) and {processed_rows} row(s) in {execution_time:.2f} seconds")
    print(f"Throughput: {throughput:.2f} rows/sec")
    return build_return_dict_ingestion(slice_path = slice_path, operation = IN_MEMORY, ntest = ntest, processed_files = processed_files, processed_rows = processed_rows, 
                                       execution_time = execution_time, throughput = throughput, benchmark_start_timestamp =  benchmark_start_timestamp.isoformat(),
                                       benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), resource_stats = resource_stats,
                                       msg = "Ingestion finished successfully")

##### Benchmark ingestion using chunks

In [16]:
# Benchmark ingestion performed in chunks using Pandas library, only reading files by iterating
# through all CSV files in the given slice directory, counting rows, measuring execution time,
# throughput and other metrics.
# Formal parameters:
# slice_path: path to the directory containing CSV files for the slice
# ntest: test number
# logger: Logger instance for reporting progress and errors
# chunksize: size of the chunk used to process the data
# Returns:
# benchmark results including slice path, processed files, processed rows, execution time, throughput, timestamps, 
# resource statistics and a message
def benchmark_pandas_ingestion_read_only_chunk(
    slice_path: str | Path, 
    ntest: int,
    chunksize: int = 100_000, 
    logger: logging.Logger | None = None
) -> dict[str, Any]:

    if logger is None:
        logger = setup_logger()
        
    timer = Timer()
    monitor = ResourceMonitor()
    timer.start("total")
    monitor.start()
    benchmark_start_timestamp = Timer.get_timestamp()

    slice_path = Path(slice_path)
    files = sorted(slice_path.glob(CSV_PATTERN))

    if len(files) == 0:
        
        log_error_msg(logger, f"No CSV files found in folder: {slice_path}")
        timer.stop("total")
        benchmark_end_timestamp = Timer.get_timestamp()
        resource_stats = monitor.stop()
        execution_time = timer.duration("total")
        return build_return_dict_ingestion(slice_path = slice_path, operation = CHUNKING, ntest = ntest, processed_files = 0, processed_rows = 0, 
                                           execution_time = execution_time, throughput = 0.0, benchmark_start_timestamp =  benchmark_start_timestamp.isoformat(),
                                           benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), resource_stats = resource_stats,
                                           msg = MSG_ERROR, chunksize = chunksize)

    log_info_msg(logger, "NYC Yellow trip taxi dataset")
    processed_rows = 0
    processed_files = 0

    for f in files:
        print(f"Reading {f}...")
        try:
            chunker_iter = pd.read_csv(
                f,
                chunksize = chunksize,
                low_memory = False
            )

            for chunk in chunker_iter:
                processed_rows += len(chunk)
                del chunk

        except Exception as e:
            log_error_msg(logger, f"Failed to open file {f} as chunked CSV: {type(e).__name__}: {e}")

            timer.stop("total")
            benchmark_end_timestamp = Timer.get_timestamp()
            resource_stats = monitor.stop()
            execution_time = timer.duration("total")
            throughput = processed_rows / execution_time if execution_time > 0 else 0
            return build_return_dict_ingestion(slice_path = slice_path, operation = CHUNKING, ntest = ntest, processed_files = processed_files, processed_rows = processed_rows, 
                                               execution_time = execution_time, throughput = throughput, benchmark_start_timestamp =  benchmark_start_timestamp.isoformat(),
                                               benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), resource_stats = resource_stats,
                                               msg = MSG_ERROR, chunksize = chunksize)
            
        gc.collect()    
        processed_files += 1
    
    timer.stop("total")
    benchmark_end_timestamp = Timer.get_timestamp()
    resource_stats = monitor.stop()
    
    execution_time = timer.duration("total")
    throughput = processed_rows / execution_time if execution_time > 0 else 0.0
    
    print(f"Pandas ingested {processed_files} file(s) and {processed_rows} row(s) in {execution_time:.2f} seconds")
    print(f"Throughput: {throughput:.2f} rows/sec")
    return build_return_dict_ingestion(slice_path = slice_path, operation = CHUNKING, ntest = ntest, processed_files = processed_files, processed_rows = processed_rows, 
                                       execution_time = execution_time, throughput = throughput, benchmark_start_timestamp =  benchmark_start_timestamp.isoformat(),
                                       benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), resource_stats = resource_stats,
                                       msg = "Ingestion finished successfully", chunksize = chunksize)

In [160]:
# Benchmark all  the slices
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
for slice_name, values in results.items():
    print(values["slice_path"])
    if (values["mode"] == IN_MEMORY):
        result = benchmark_pandas_ingestion_read_only(slice_path = values["slice_path"], ntest = 1, logger = logger)
        print(result)
    elif (values["mode"] == CHUNKING):
        result = benchmark_pandas_ingestion_read_only_chunk(slice_path = values["slice_path"], ntest = 1, logger = logger)
        print(result)
    else: raise ValueError(f"Unknown mode: {values['mode']}")

../data/CSV/raw-data/1st-slice/
Reading ../data/CSV/raw-data/1st-slice/yellow_tripdata_2025-07.csv...
Pandas ingested 1 file(s) and 3898963 row(s) in 12.70 seconds
Throughput: 306915.72 rows/sec
{'stage': 'ingestion', 'operation': 'in-memory', 'test_number': 1, 'slice_path': '../data/CSV/raw-data/1st-slice', 'processed_files': 1, 'processed_rows': 3898963, 'execution_time_sec': 12.703692660001252, 'throughput_rows_sec': 306915.7216213396, 'chunksize': 0, 'dataset_size_mib': 402.3603820800781, 'benchmark_start_timestamp': '2026-04-10T12:50:13.246530+01:00', 'benchmark_end_timestamp': '2026-04-10T12:50:25.950271+01:00', 'avg_rss_mib': 1148.5472790948277, 'peak_rss_mib': 2727.5703125, 'memory_percent_of_total': 34.76521735666887, 'avg_process_cpu_percent': 112.34827586206896, 'peak_process_cpu_percent': 174.1, 'avg_system_cpu_per_core_percent': [9.768965517241378, 53.64482758620689, 12.410344827586206, 39.193103448275856], 'peak_system_cpu_per_core_percent': [80.0, 100.0, 100.0, 100.0], '

In [16]:
result = benchmark_pandas_ingestion_read_only(slice_path = "../data/CSV/raw-data/1st-slice/", ntest = 1, logger = logger)
print(result["msg"])

Reading ../data/CSV/raw-data/1st-slice/yellow_tripdata_2025-07.csv...
Pandas ingested 1 file(s) and 3898963 row(s) in 115.78 seconds
Throughput: 33675.03 rows/sec
Ingestion finished successfully


In [155]:
result = benchmark_pandas_ingestion_read_only(slice_path = "../data/CSV/raw-data/2nd-slice/", ntest = 1, logger = logger)
print(result["msg"])

Reading ../data/CSV/raw-data/2nd-slice/yellow_tripdata_2025-05.csv...
Reading ../data/CSV/raw-data/2nd-slice/yellow_tripdata_2025-06.csv...
Reading ../data/CSV/raw-data/2nd-slice/yellow_tripdata_2025-07.csv...
Pandas ingested 3 file(s) and 12813768 row(s) in 35.06 seconds
Throughput: 365447.78 rows/sec
Ingestion finished successfully


In [156]:
import json
print(json.dumps(result, indent = 4))

{
    "stage": "ingestion",
    "operation": "in-memory",
    "test_number": 1,
    "slice_path": "../data/CSV/raw-data/2nd-slice",
    "processed_files": 3,
    "processed_rows": 12813768,
    "execution_time_sec": 35.06319807900036,
    "throughput_rows_sec": 365447.7829184176,
    "chunksize": 0,
    "dataset_size_mib": 1321.7953338623047,
    "benchmark_start_timestamp": "2026-04-10T12:32:00.615830+01:00",
    "benchmark_end_timestamp": "2026-04-10T12:32:35.679067+01:00",
    "avg_rss_mib": 1430.3550112612613,
    "peak_rss_mib": 3090.53125,
    "memory_percent_of_total": 39.39146505644024,
    "avg_process_cpu_percent": 110.5765765765766,
    "peak_process_cpu_percent": 169.8,
    "avg_system_cpu_per_core_percent": [
        6.208108108108108,
        20.568468468468446,
        1.163063063063063,
        80.45135135135133
    ],
    "peak_system_cpu_per_core_percent": [
        100.0,
        100.0,
        18.2,
        100.0
    ],
    "read_mib": 1320.8046875,
    "write_mib":

In [157]:
result = benchmark_pandas_ingestion_read_only_chunk(slice_path = "../data/CSV/raw-data/2nd-slice/", ntest = 2, logger = logger)
print(result["msg"])

Reading ../data/CSV/raw-data/2nd-slice/yellow_tripdata_2025-05.csv...
Reading ../data/CSV/raw-data/2nd-slice/yellow_tripdata_2025-06.csv...
Reading ../data/CSV/raw-data/2nd-slice/yellow_tripdata_2025-07.csv...
Pandas ingested 3 file(s) and 12813768 row(s) in 20.64 seconds
Throughput: 620747.31 rows/sec
Ingestion finished successfully


In [158]:
import json
print(json.dumps(result, indent = 4))

{
    "stage": "ingestion",
    "operation": "chunking",
    "test_number": 2,
    "slice_path": "../data/CSV/raw-data/2nd-slice",
    "processed_files": 3,
    "processed_rows": 12813768,
    "execution_time_sec": 20.64248647400018,
    "throughput_rows_sec": 620747.3124005346,
    "chunksize": 100000,
    "dataset_size_mib": 1321.7953338623047,
    "benchmark_start_timestamp": "2026-04-10T12:36:00.275871+01:00",
    "benchmark_end_timestamp": "2026-04-10T12:36:20.914135+01:00",
    "avg_rss_mib": 310.72061353211006,
    "peak_rss_mib": 319.86328125,
    "memory_percent_of_total": 4.07693119627821,
    "avg_process_cpu_percent": 109.83394495412847,
    "peak_process_cpu_percent": 182.2,
    "avg_system_cpu_per_core_percent": [
        5.722935779816515,
        98.42385321100917,
        0.21009174311926607,
        1.103669724770642
    ],
    "peak_system_cpu_per_core_percent": [
        16.7,
        100.0,
        8.3,
        33.3
    ],
    "read_mib": 0.125,
    "write_mib": 0.

#### Cleaning, transformation & Profiling Stage

In [17]:
# Extract year and month from a filename using the pattern YYYY-MM
def extract_year_month_from_filename(
    filename: str | Path
) -> tuple[int | None, int | None, str]:
    filename = str(filename)
    msg = ""
    match = re.search(r"(\d{4})-(\d{2})", filename)
    if not match:
        msg = f"Could not extract year-month from filename: {filename}"
        return None, None, msg
        
    return int(match.group(1)), int(match.group(2)), msg

In [18]:
# Check if a pandas Dataframe is empty
def is_empty(df: pd.DataFrame) -> bool:
    return df.empty

In [19]:
# Rename columns of a DataFrame according to a predefined mapping
def rename_columns(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:

    COLUMN_RENAME_MAP = {
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "passenger_count": "passenger_count",
        "trip_distance": "trip_distance",
        "RatecodeID": "rate_code_id",
        "store_and_fwd_flag": "store_and_forward_flag",
        "PULocationID": "pickup_location_id",
        "DOLocationID": "dropoff_location_id",
        "payment_type": "payment_type_id",
        "fare_amount": "fare_amount",
        "extra": "extra_charge",
        "mta_tax": "metered_rate_tax",
        "tip_amount": "tip_amount",
        "tolls_amount": "tolls_amount",
        "improvement_surcharge": "improvement_surcharge",
        "total_amount": "total_amount",
        "congestion_surcharge": "congestion_surcharge",
        "Airport_fee": "airport_fee",
        "cbd_congestion_fee": "cbd_congestion_fee",
        "year": "year",
        "month": "month",
        # New fields
        "currency": "currency", 
        "trip_distance_unit": "trip_distance_unit",
        "suspicious": "suspicious"
    }
    
    df = df.rename(columns = COLUMN_RENAME_MAP)
    
    return df, "Rename was performed successfully"

In [20]:
# Builds a set of column names from a DataFrame
def build_columns_schema(df: pd.DataFrame) -> tuple[set[str], str]:
    cols = set(df.columns)
    return cols, "Schema was built successfully"

In [21]:
# Align a DataFrame to a predefined master schema and reports missing and unexpected columns.
# Adds missing columns with null values, drops unexpected columns and reorders columns to match
# the mater schema
# Parameters:
# df: Input DataFrame
# cols: set of column names present in the DataFrame
# Returns:
# aligned DataFrame and message describing missing and unexpected columns
def align_schema(
    df: pd.DataFrame, 
    cols: set[str]
) -> tuple[pd.DataFrame, str]:

    MASTER_SCHEMA = [
        "vendor_id",
        "pickup_datetime",
        "dropoff_datetime",
        "passenger_count",
        "trip_distance",
        "rate_code_id",
        "store_and_forward_flag",
        "pickup_location_id",
        "dropoff_location_id",
        "payment_type_id",
        "fare_amount",
        "extra_charge",
        "metered_rate_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "total_amount",
        "congestion_surcharge",
        "airport_fee",
        "cbd_congestion_fee",
        # New fields
        "year",
        "month",
        "currency", 
        "trip_distance_unit",
        "suspicious"
    ]

    unexpected_cols = set()
    missing_cols = set()
    
    for c in cols:
        if c not in MASTER_SCHEMA:
            unexpected_cols.add(c)

    for c in MASTER_SCHEMA:
        if c not in cols:
            df[c] = pd.NA
            missing_cols.add(c)

    df = df[MASTER_SCHEMA].copy()

    if missing_cols:
        msg_missing_cols = f"Missing columns added as nulls: {sorted(missing_cols)}"
    else:
        msg_missing_cols = "No missing columns detected"

    if unexpected_cols:
        msg_unexpected_cols = f"Unexpected columns detected: {sorted(unexpected_cols)}"
    else:
        msg_unexpected_cols = "No unexpected columns detected"
        
    msg = f"{msg_missing_cols}\n{msg_unexpected_cols}"
    
    return df, msg

In [22]:
# Compare DataFrame column dtypes against the master schema
# Parameters:
# df: input DataFrame
# cols: set of column names present in the DataFrame
# Returns:
# status message containing the Dictionary of mismatches 
def get_type_mismatches(
    df: pd.DataFrame, 
    cols: set[str]
) -> str:
    
    MASTER_SCHEMA_TYPES = {
        "vendor_id": "UInt8",
        "pickup_datetime": "datetime64[ns]",
        "dropoff_datetime": "datetime64[ns]",
        "passenger_count": "UInt8",
        "trip_distance": "Float32",
        "rate_code_id": "UInt8",
        "store_and_forward_flag": "category",
        "pickup_location_id": "Int16",
        "dropoff_location_id": "Int16",
        "payment_type_id": "UInt8",
        "fare_amount": "Float32",
        "extra_charge": "Float32",
        "metered_rate_tax": "Float32",
        "tip_amount": "Float32",
        "tolls_amount": "Float32",
        "improvement_surcharge": "Float32",
        "total_amount": "Float32",
        "congestion_surcharge": "Float32",
        "airport_fee": "Float32",
        "cbd_congestion_fee": "Float32",
        # New fields
        "year": "Int16",
        "month": "UInt8",
        "currency": "category", 
        "trip_distance_unit": "category",
        "suspicious": "boolean"
    }

    mismatches = {}
    
    for col, expected_type in MASTER_SCHEMA_TYPES.items():
        if col in cols:
            actual_type = str(df[col].dtype)
            if actual_type != expected_type:
                mismatches[col] = {
                    "expected": expected_type,
                    "actual": actual_type
                }

    if mismatches:
        msg = f"Type mismatches detected in {len(mismatches)} column(s): {mismatches}"
    else:
        msg = "No type mismatches detected"

    return msg

In [23]:
# Enforce the master schema for Yellow Taxi data by cleaning,
# and casting columns to their expected types
# Parameters:
# df: input DataFrame
# cols: set of column names present in the DataFrame
# Returns:
# DataFrame with enforced schema and status message
def enforce_master_schema(
    df: pd.DataFrame, 
    cols: set[str]
) -> tuple[pd.DataFrame, str]:

    # Integer columns
    for c in ["vendor_id", "passenger_count", "rate_code_id", "payment_type_id", "month"]:
        if c in cols:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("UInt8")

    # Greater Integer columns
    for c in ["pickup_location_id", "dropoff_location_id", "year"]:
        if c in cols:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int16") 

    # Float columns
    for c in ["trip_distance", "fare_amount", "extra_charge", "metered_rate_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount", 
              "congestion_surcharge", "airport_fee", "cbd_congestion_fee"]:
        if c in cols:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Float32")

    # Datetime columns
    for c in ["pickup_datetime", "dropoff_datetime"]:
        if c in cols:
            df[c] = pd.to_datetime(df[c], errors="coerce")

    # Categorical columns
    if "store_and_forward_flag" in cols:
        df["store_and_forward_flag"] = df["store_and_forward_flag"].str.strip().str.upper()
        df["store_and_forward_flag"] = pd.Categorical(df["store_and_forward_flag"], categories = ["Y", "N", "UNKNOWN"])

    if "currency" in cols:
        df["currency"] = pd.Categorical(df["currency"], categories = ["USD"])

    if "trip_distance_unit" in cols:
        df["trip_distance_unit"] = pd.Categorical(df["trip_distance_unit"], categories = ["Miles"])

    if "suspicious" in cols:
        df["suspicious"] = df["suspicious"].astype("boolean")

    return df, "Master schema enforcement completed"

In [24]:
# Applies the master schema to a DataFrame by:
# - renaming columns
# - building and logging schema before alignment
# - aligning columns to the master schema
# - checking type mismatches before enforcement
# - enforcing master schema
# - checking type mismatches after enforcement
def apply_master_schema(
    df: pd.DataFrame, 
    logger: logging.Logger | None = None
) -> pd.DataFrame:
    
    if logger is None:
        logger = setup_logger()

    # Rename the columns
    df, msg = rename_columns(df)
    log_info_msg(logger, msg)
    
    # schema before alignment
    original_cols, msg = build_columns_schema(df)
    
    log_info_msg(logger, f"Schema before alignment:\n{msg}")
    
    # align columns
    df, msg = align_schema(df, original_cols)

    log_info_msg(logger, msg)
    
    # schema after alignment
    aligned_cols, msg = build_columns_schema(df)

    log_info_msg(logger, f"Schema after alignment:\n{msg}")
    
    # check types before enforcement
    msg = get_type_mismatches(df, aligned_cols)

    log_info_msg(logger, f"Type mismatches before enforcement:\n{msg}")
    
    # enforce schema
    df, msg = enforce_master_schema(df, aligned_cols)

    log_info_msg(logger, msg)
    
    # check types after enforcement
    msg = get_type_mismatches(df, aligned_cols)

    log_info_msg(logger, f"Type mismatches after enforcement:\n{msg}")

    return df

In [25]:
# Impute missing values for selected columns:
# - tip_amount -> 0
# - passenger_count -> 1
# - store_and_forward_flag -> "UNKNOWN"
def impute_missing_values(
    df: pd.DataFrame
) -> tuple[pd.DataFrame, str]:

    df = df.copy()
    
    tip_missing = 0
    passenger_missing = 0
    store_and_forward_flag_missing = 0
    
    if "tip_amount" in df.columns:
        tip_missing = df["tip_amount"].isna().sum()
        df["tip_amount"] = df["tip_amount"].fillna(0)

    if "passenger_count" in df.columns:
        passenger_missing = df["passenger_count"].isna().sum()
        df["passenger_count"] = df["passenger_count"].fillna(1)
        
    if "store_and_forward_flag" in df.columns:
        store_and_forward_flag_missing = df["store_and_forward_flag"].isna().sum()
        df["store_and_forward_flag"] = df["store_and_forward_flag"].fillna("UNKNOWN")

    msgs = []

    if tip_missing == 1:
        msgs.append("1 missing tip_amount value was imputed with 0")
    elif tip_missing > 1:
        msgs.append(f"{tip_missing} missing tip_amount values were imputed with 0")

    if passenger_missing == 1:
        msgs.append("1 missing passenger_count value was imputed with 1")
    elif passenger_missing > 1:
        msgs.append(f"{passenger_missing} missing passenger_count values were imputed with 1")

    if store_and_forward_flag_missing == 1:
        msgs.append("1 missing store_and_forward_flag value was imputed with UNKNOWN")
    elif store_and_forward_flag_missing > 1:
        msgs.append(f"{store_and_forward_flag_missing} missing store_and_forward_flag values were imputed with UNKNOWN")
        
    if not msgs:
        msg = "No missing values were imputed"
    else:
        msg = ", ".join(msgs)

    return df, msg

In [26]:
# Imputes missing total_amount values by summing available monetary fields.
# A row is imputed when:
# 1. total_amount is missing (NaN) and
# 2. at least one of the individual monetary fields is present.
def impute_missing_total_amount(
    df: pd.DataFrame
) -> tuple[pd.DataFrame, str]:

    df = df.copy()

    AMOUNT_COLS = ["fare_amount", "extra_charge", "metered_rate_tax", "tip_amount", "tolls_amount",                    
                     "improvement_surcharge", "congestion_surcharge", "airport_fee",               
                     "cbd_congestion_fee"]

    valid_cols = [c for c in AMOUNT_COLS]

    fill_mask = (
        df["total_amount"].isna() &
        df[valid_cols].notna().any(axis = 1)
    )
    
    expected_total = df.loc[fill_mask, valid_cols].fillna(0).sum(axis = 1)

    df.loc[fill_mask, "total_amount"] = expected_total

    filled = int(fill_mask.sum())

    if filled == 0:
        msg = "No total amount values were imputed"
    elif filled == 1:
        msg = "1 total amount value was imputed summing all the monetary values"
    else:
        msg = f"{filled} total amount values were imputed summing all the currency values"

    return df, msg

In [27]:
# Remove rows where all columns are missing
# Parameters:
# df: input DataFrame
# Returns:
# Clean DataFrame, DataFrame containing removed row(s) and status message
def remove_rows_all_columns_missing(
    df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, str] :

    invalid_mask = df.isna().all(axis = 1)
    df_removed_rows = df[invalid_mask].copy()
    
    df = df[~invalid_mask].copy()

    removed = len(df_removed_rows)
    
    if removed == 0:
        msg = "0 rows were dropped because no row had all columns null"
    elif removed == 1:
        msg = "1 row was dropped because all its columns were null"
    else: 
        msg = f"{removed - len(df)} were dropped because all their columns were null" 

    return df, df_removed_rows, msg    

In [28]:
# Remove rows where all specified critical columns are missing
# Parameters:
# df: input DataFrame
# critical_cols: columns considered critical. 
# Returns:
# Cleaned DataFrame, DataFrame of removed row(s) and status message
def remove_rows_with_missing_critical_columns(
    df: pd.DataFrame, 
    critical_cols: list[str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    
    if critical_cols is None:
        return df, pd.DataFrame(), "No critical columns provided, no rows were removed"

    invalid_mask = df[critical_cols].isna().all(axis = 1)
    df_removed_rows = df[invalid_mask].copy()
    
    df = df[~invalid_mask].copy()
    removed = len(df_removed_rows)

    if removed == 0:
        msg = "No rows were dropped, because all rows had at least one critical field present"
    elif removed == 1:
        msg = "1 row was dropped because all critical fields were missing"
    else:
        msg = f"{removed} rows were dropped because all critical fields were missing"
    
    return df, df_removed_rows, msg

In [29]:
# Remove rows with invalid timestamps
# A row is considered valid if:
# - pickup_datetime and dropoff_datetime are not null
# - pickup_datetime < dropoff_dateimte
# Parameters:
# df: input DataFrame
# Returns:
# Cleaned DataFrame, DataFrame of removed row(s) and status message
def remove_invalid_timestamps(
    df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, str]:

    not_nat_mask = (df["pickup_datetime"].notna() & df["dropoff_datetime"].notna())  

    order_mask = df["pickup_datetime"].lt(df["dropoff_datetime"])

    valid_mask = not_nat_mask & order_mask

    df_removed_rows = df[~valid_mask].copy()
    
    df = df[valid_mask].copy()

    removed = len(df_removed_rows)
    
    if removed == 0:
        msg = "No invalid timestamps were found"
    elif removed == 1:
        msg = "1 row had invalid timestamp(s)"
    else:
        msg = f"{removed} rows had invalid timestamps"
        
    return df, df_removed_rows, msg

In [30]:
# Validate year/month leakage in a chunk DataFrame by checking that pickup_datetime
# matches the single year-month pair declared in the chunk
# A row is removed when:
# - pickup_datetime.year != year 
# - pickup_datetime.month != month 
# - pickup_datetime, year, month is missing 
# Parameters:
# df: input DataFrame
# Returns:
# Cleaned DataFrame, DataFrame of leaked row(s) and status message
def validate_year_month_leakage_chunk(
    df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, str]:

    if df["year"].isna().all() or df["month"].isna().all():
        empty = pd.DataFrame(columns = df.columns)
        return empty, empty.copy(), f"Cannot determine expected year/month pair (all year or/and month values are null)"

    year_values = df["year"].dropna().unique()
    month_values = df["month"].dropna().unique()

    if len(year_values) != 1 or len(month_values) != 1:
        empty = pd.DataFrame(columns = df.columns)
        return empty, empty.copy(), "Chunk has inconsistent year and month values"

    expected_year = year_values[0]
    expected_month = month_values[0]

    mask = (
        df["pickup_datetime"].notna() &
        df["year"].notna() &
        df["month"].notna() &
        df["pickup_datetime"].dt.year.eq(expected_year) &
        df["pickup_datetime"].dt.month.eq(expected_month)
    )

    leaked_rows = df[~mask].copy()
    valid_rows = df[mask].copy()

    nleaked = len(leaked_rows)

    if nleaked == 0:
        leak_msg = "No leaked rows were detected"
    elif nleaked == 1:
        leak_msg = "1 leaked row was detected"
    else:
        leak_msg = f"{nleaked} leaked rows were detected"
    
    msg = f"Perform leakage validation for: {int(expected_year)}-{int(expected_month):02d}\n{leak_msg}"
    
    return valid_rows, leaked_rows, msg

In [31]:
# Validate year/month leakage in a DataFrame by checking that pickup_datetime
# matches the single year-month pair declared in the chunk
# A row is removed when:
# - pickup_datetime.year != year 
# - pickup_datetime.month != month 
# - pickup_datetime, year, month is missing 
# Parameters:
# df: input DataFrame
# Returns:
# Cleaned DataFrame, DataFrame of leaked row(s) and status message
def validate_year_month_leakage_in_memory(
    df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    
    if df["year"].isna().all() or df["month"].isna().all():
        empty = pd.DataFrame(columns = df.columns)
        return empty, empty.copy(), f"Cannot determine expected year/month pairs (all year or/and month values are null)"
        
    mask = (
        df["pickup_datetime"].notna() &
        df["year"].notna() &
        df["month"].notna() &
        (df["pickup_datetime"].dt.year == df["year"]) &
        (df["pickup_datetime"].dt.month == df["month"])
    )

    leaked_rows = df[~mask].copy()
    valid_rows = df[mask].copy()

    expected_years_months = sorted(set(zip(df["year"].dropna(), df["month"].dropna())))

    header_msg = "Perform leakage validation for the following year-month pairs:\n"

    pairs = "\n".join(f"- {int(y)}-{int(m):02d}" for y, m in expected_years_months)

    nleaked = len(leaked_rows)

    if nleaked == 0:
        leak_msg = "No leaked rows were detected"
    elif nleaked == 1:
        leak_msg = "1 leaked row was detected"
    else:
        leak_msg = f"{nleaked} leaked rows were detected"

    msg = f"{header_msg}{pairs}\n{leak_msg}"

    return valid_rows, leaked_rows, msg

In [32]:
# Removes rows with negative trip_distance values
def remove_negative_trip_distance(
    df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, str]:

    valid_mask = df["trip_distance"].ge(0) | df["trip_distance"].isna()

    df_removed_rows = df[~valid_mask].copy()
    
    df = df[valid_mask].copy()

    removed = len(df_removed_rows)
    
    if removed == 0:
        msg = "No rows were removed due to negative trip distance"
    elif removed == 1:
        msg = "1 row was removed because it had a negative trip distance"
    else:
        msg = f"{removed} rows were removed because they had negative trip distances"
    
    return df, df_removed_rows, msg    

In [33]:
# Removes rows with negative passenger_count values
def remove_unrealistic_passenger_count(
    df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, str]:

    valid_mask = df["passenger_count"].ge(0) | df["passenger_count"].isna()
    df_removed_rows = df[~valid_mask].copy()

    df = df[valid_mask].copy()
    
    removed = len(df_removed_rows)
    
    if removed == 0:
        msg = "No rows were removed due to negative passenger count"
    elif removed == 1:
        msg = "1 row was removed because it had a negative passenger count"
    else:
        msg = f"{removed} rows were removed because they had negative passenger counts"
    
    return df, df_removed_rows, msg     

In [34]:
# This function evaluates several conditions to identify rows where monetary
# values appear inconsistent or incomplete. A row is flagged as suspicious if:
# 1. The total_amount differs from the sum of all individual monetary fields/columns
#    by more than the specified tolerance
# 2. All monetary fields/columns are missing 
# 3. The total_amount field/column is missing
# 4. The tip_amount field/column is missing
# Parameters:
# df: input DataFrame
# tolerance: tolerance to compare with the difference between total amount and all the other fields//columns
# Returns:
# Dataframe with flag, suspicious row(s) and message
def add_suspicious_flag_amount_values_currency(
    df: pd.DataFrame, 
    tolerance: float = 2.0
) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    
    df = df.copy()
    
    suspicious_mask = pd.Series(False, index = df.index)

    # Inconsistent total amount
    AMOUNT_COLS = ["fare_amount", "extra_charge", "metered_rate_tax", "tip_amount", "tolls_amount",                    
                    "improvement_surcharge", "congestion_surcharge", "airport_fee",               
                    "cbd_congestion_fee"]

    expected_total = df[AMOUNT_COLS].fillna(0).sum(axis = 1)
    diff = (df["total_amount"] - expected_total).abs()
    suspicious_mask = suspicious_mask | (df["total_amount"].notna() & diff.gt(tolerance))

    # All monetary fields are null
    all_null_mask = df[AMOUNT_COLS].isna().all(axis = 1)
    suspicious_mask = suspicious_mask | all_null_mask

    # Missing total amount
    missing_total_mask = df["total_amount"].isna()
    suspicious_mask = suspicious_mask | missing_total_mask

    # Missing tip amount and payment type is card
    missing_tip_mask = (df["payment_type_id"] == 1) & df["tip_amount"].isna()
    suspicious_mask = suspicious_mask | missing_tip_mask
    
    # Add column to the original dataframe
    df["suspicious"] = suspicious_mask

    # Check how many suspicious rows there are
    df_suspicious_rows = df[df["suspicious"]].copy()

    nsuspicious = len(df_suspicious_rows)

    if nsuspicious == 0:
        msg = "No suspicious monetary rows were flagged"
    elif nsuspicious == 1:
        msg = "1 suspicious row was flagged as suspicious"
    else:
        msg = f"{nsuspicious} rows were flagged as suspicious"

    return df, df_suspicious_rows, msg

In [35]:
# Identifies trips with unrealistic speeds by computing trip duration in minutes,
# deriving speed in mph, and removing rows where the speed exceeds a
# specified maximum threshold.
# Parameters:
# df: input DataFrame
# max_speed_mph: maximum allowed speed in mph
# Results:
# cleaned DataFrame, removed row(s) and status message
def remove_unrealistic_trip_speed(
    df: pd.DataFrame, 
    max_speed_mph: float = 120
) -> tuple[pd.DataFrame, pd.DataFrame, str]:

    df = df.copy()
    
    df["trip_duration_min"] = (
        df["dropoff_datetime"] - df["pickup_datetime"]
    ).dt.total_seconds() / 60 # converted to minutes

    valid_duration = df["trip_duration_min"].gt(0)

    df["avg_speed_mph"] = pd.NA

    df.loc[valid_duration, "avg_speed_mph"] =  (
        df.loc[valid_duration, "trip_distance"] / 
        (df.loc[valid_duration, "trip_duration_min"] / 60)) # converted to hours

    valid_mask = (
        df["avg_speed_mph"].isna() |
        (df["avg_speed_mph"] <= max_speed_mph))

    df_removed_rows = df[~valid_mask].copy() 
    
    df = df[valid_mask].copy()

    df = df.drop(columns = ["trip_duration_min", "avg_speed_mph"], errors = "ignore")
    
    removed = len(df_removed_rows)
    
    if removed == 0:
        msg = "No rows were removed due to unrealistic trip speeds"
    elif removed == 1:
        msg = "1 row was removed due to unrealistic trip speed"
    else:
        msg = f"{removed} rows were removed due to unrealistic trip speeds"
    
    return df, df_removed_rows, msg

In [36]:
# Profiles by computing missing value count and percentage per column
# and returning a DataFrame sorted in descending order of the number of missing values
def profile_missing_values(
    df: pd.DataFrame
) -> pd.DataFrame:

    total_rows = len(df)
    missing_count = df.isna().sum()
    missing_percent = ((missing_count / total_rows) * 100).round(2)

    result = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent
    })
    
    return result.sort_values(by = "missing_count", ascending = False)

In [37]:
# Profiles vendor-specific differences by computing descriptive statistics
# (count, mean, median, std, min, max) for numeric fields grouped by vendor_id.
def profile_vendor_specific_differences(
    df: pd.DataFrame
) -> pd.DataFrame:
    
    cols = ["trip_distance", "fare_amount", "tip_amount", "total_amount"]
    summary = df.groupby("vendor_id")[cols].agg(["count", "mean", "median", "std", "min", "max"])
    summary.columns = [
        f"{col}_{stat}" for col, stat in summary.columns
    ]
    return summary

In [38]:
# Update missing value counts using one chunk
# Parameters:
# df: input DataFrame
# missing_counts_total: total of missing counts per column
# Returns:
# update missing counts
def update_profile_missing_values(
    df: pd.DataFrame,
    missing_counts_total: pd.Series | None = None
) -> pd.Series:
    chunk_missing = df.isna().sum()

    if missing_counts_total is None:
        return chunk_missing

    return missing_counts_total.add(chunk_missing, fill_value = 0).astype("int64")

In [39]:
# Build final missing values profile 
# Parameters:
# missing_counts_total: total of missing counts per column
# Total number of rows across all chunks
# Returns:
# DataFrame with missing_count and missing_percent
def finalise_profile_missing_values(
    missing_counts_total: pd.Series | None,
    total_rows: int
) -> pd.DataFrame:
    if missing_counts_total is None or missing_counts_total.empty:
        return pd.DataFrame(columns=["missing_count", "missing_percent"])

    denominator = total_rows if total_rows > 0 else 1

    result = pd.DataFrame({
         "missing_count": missing_counts_total,
         "missing_percent": ((missing_counts_total / denominator) * 100).round(2)
    })

    return result.sort_values(by = "missing_count", ascending = False)

In [40]:
# Applies the full cleaning and transformation pipeline to a slice of data.
# The pipeline sequentially:
# - removes rows with all columns missing
# - removes rows wit all critical columns missing
# - removes invalid timestamps
# - removes year/month leakage
# - removes negative trip distances
# - removes unrealistic passenger counts
# - removes unrealistic trip speeds
# - flags suspicious monetary rows and save a sample
# - imputes selected missing values
# - imputes missing total amount values
# Removed rows are written to CSV files under removed_slice_path
# Processing stops early if the DataFrame becomes empty
# Parameters:
# df: input DataFrame
# removed_slice_path: directory where removed-rows and suspicious-rows will be saved
# logger: Logger instance for reporting progress and errors
# process_type: leakage validation mode
# header: if to write CSV headers when appending output to files
# Returns:
# cleaned DataFrame
def cleaning_transformation(
    df: pd.DataFrame, 
    removed_slice_path: str | Path, 
    logger: logging.Logger | None = None, 
    process_type: Literal[IN_MEMORY, CHUNKING] = IN_MEMORY, 
    header: bool = False
) -> pd.DataFrame:
    
    if logger is None:
        logger = setup_logger()

    removed_slice_path = Path(removed_slice_path)
    df, df_removed_rows, msg = remove_rows_all_columns_missing(df)
    log_info_msg(logger, msg)
    if not is_empty(df_removed_rows):
        path = removed_slice_path / "rows_all_columns_missing.csv"
        log_info_msg(logger, f"Review file: {path} to check the removed rows")
        df_removed_rows.to_csv(path, mode = "a", index = False, header = header)
    
    if not is_empty(df):
        df, df_removed_rows, msg = remove_rows_with_missing_critical_columns(df, ["pickup_datetime", "dropoff_datetime", "trip_distance", "pickup_location_id",  
                                                      "dropoff_location_id", "fare_amount"])
        log_info_msg(logger, msg)
        if not is_empty(df_removed_rows):
            path = removed_slice_path / "rows_missing_critical_columns.csv"
            log_info_msg(logger, f"Review file: {path} to check the removed rows")
            df_removed_rows.to_csv(path, mode = "a", index = False, header = header)
        
    if not is_empty(df):
        df, df_removed_rows, msg =  remove_invalid_timestamps(df)
        log_info_msg(logger, msg)
        if not is_empty(df_removed_rows):
            path = removed_slice_path / "invalid_timestamps.csv"
            log_info_msg(logger, f"Review file: {path} to check the removed rows")
            df_removed_rows.to_csv(path, mode = "a", index = False, header = header)

    if not is_empty(df):

        if process_type == IN_MEMORY:
            df, leaked_df, msg = validate_year_month_leakage_in_memory(df)
            log_info_msg(logger, msg)
            if not is_empty(leaked_df):
                path = removed_slice_path / "month_leaked_rows.csv"
                log_info_msg(logger, f"Review file: {path} to check the leaked rows")
                leaked_df.to_csv(path, mode = "a", index = False, header = header)
            
        elif process_type == CHUNKING:
            df, leaked_df, msg = validate_year_month_leakage_chunk(df)
            log_info_msg(logger, msg)
            if not is_empty(leaked_df):
                path = removed_slice_path / "month_leaked_rows.csv"
                log_info_msg(logger, f"Review file: {path} to check the leaked rows")
                leaked_df.to_csv(path, mode = "a", index = False, header = header)

        else: log_info_msg(logger, f"Process type {process_type} does not exist")
    
    if not is_empty(df):
        df, df_removed_rows, msg = remove_negative_trip_distance(df)
        log_info_msg(logger, msg)
        if not is_empty(df_removed_rows):
            path = removed_slice_path / "negative_trip_distance.csv"
            log_info_msg(logger, f"Review file: {path} to check the removed rows")
            df_removed_rows.to_csv(path, mode = "a", index = False, header = header)

    if not is_empty(df):
        df, df_removed_rows, msg = remove_unrealistic_passenger_count(df)
        log_info_msg(logger, msg)
        if not is_empty(df_removed_rows):
            path = removed_slice_path / "unrealistic_passenger_count.csv"
            log_info_msg(logger, f"Review file: {path} to check the removed rows")
            df_removed_rows.to_csv(path, mode = "a", index = False, header = header)

    if not is_empty(df):
        df, df_removed_rows, msg = remove_unrealistic_trip_speed(df)
        log_info_msg(logger, msg)
        if not is_empty(df_removed_rows):
            path = removed_slice_path / "unrealistic_trip_speed.csv"
            log_info_msg(logger, f"Review file: {path} to check the removed rows")
            df_removed_rows.to_csv(path, mode = "a", index = False, header = header)

    if not is_empty(df):
        MAX_SUSPICIOUS_ROWS = 50
        df, df_suspicious, msg = add_suspicious_flag_amount_values_currency(df)
        log_info_msg(logger, msg)
        if not is_empty(df_suspicious):
            path = removed_slice_path / "suspicious_rows(NO_REMOVED_ROWS).csv"
            if len(df_suspicious) > MAX_SUSPICIOUS_ROWS:
                df_suspicious = df_suspicious.sample(MAX_SUSPICIOUS_ROWS, random_state = 42)
                log_info_msg(logger, f"{MAX_SUSPICIOUS_ROWS} suspicious rows were chosen using random sampling to be saved in the file: {path}")
            else:
                log_info_msg(logger, f"All {len(df_suspicious)} suspicious rows were saved in the file: {path}")
                
            df_suspicious.to_csv(path, mode = "a", index = False, header = header)

    if not is_empty(df):
        df, msg = impute_missing_values(df)
        log_info_msg(logger, msg) 

    if not is_empty(df):
        df, msg = impute_missing_total_amount(df)
        log_info_msg(logger, msg) 
    
    return df

##### Processing data in memory

In [41]:
# Performs ingestion of CSV files using Pandas library
# Parameters:
# slice_path: path containing CSV files to ingest
# logger: Logger instance for reporting progress and errors
# Returns:
# Ingested DataFrame, number of processed files, success flag
def pandas_ingestion(
    slice_path: str | Path, 
    logger: logging.Logger | None = None
) -> tuple[pd.DataFrame, int, bool]: 

    if logger is None:
        logger = setup_logger()

    log_info_msg(logger, "NYC Yellow trip taxi dataset")

    CURRENCY = "USD"
    DISTANCE_UNIT = "Miles"
    
    slice_path = Path(slice_path)
    files = sorted(slice_path.glob(CSV_PATTERN))

    if len(files) == 0:
        log_error_msg(logger, f"No CSV files found in folder: {slice_path}")
        return pd.DataFrame(), 0, 0, False
        
    dfs = []
    processed_files = 0
    processed_rows = 0

    for f in files:
        
        print(f"Reading {f}...")

        year, month, msg = extract_year_month_from_filename(f)

        if year is None or month is None:
            log_error_msg(logger, msg)
            return pd.DataFrame(), processed_files, processed_rows, False
            
        try:
            
            df_aux = pd.read_csv(f, low_memory = False)
            processed_rows += len(df_aux)
            df_aux["year"] = year
            df_aux["month"] = month
            df_aux["currency"] = CURRENCY
            df_aux["trip_distance_unit"] = DISTANCE_UNIT
            df_aux["suspicious"] = False
        
        except Exception as e:
            log_error_msg(logger, f"Failed to open file {f}: {type(e).__name__}: {e}")
            return pd.DataFrame(), processed_files, processed_rows, False

        dfs.append(df_aux)
        processed_files += 1

    if not dfs:
        msg = f"No valid CSV files were ingested in {slice_path}"
        log_error_msg(logger, msg)
        return pd.DataFrame(), processed_files, processed_rows, False

    try:
        
        df = pd.concat(dfs, ignore_index = True)

    except Exception as e:
        msg = f"Failed to concatenate DataFrames: {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        return pd.DataFrame(), processed_files, processed_rows, False
    
    log_info_msg(logger, "Ingestion finished successfully")
    
    return df, processed_files, processed_rows, True

In [40]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
df, total_files, total_rows, ingested = pandas_ingestion(slice_path = "../data/CSV/raw-data/1st-slice/", logger = logger)
if not ingested:
    print("Ingested failed, check pipeline.log")    

Reading ../data/CSV/raw-data/1st-slice/yellow_tripdata_2025-07.csv...
Ingested failed, check pipeline.log


In [88]:
print(f"{total_files} {total_rows}")

1 3898963


In [89]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,year,month,currency,trip_distance_unit,suspicious
0,1,2025-07-01 00:29:37,2025-07-01 00:45:30,1.0,7.30,1.0,N,138,74,1,...,1.0,54.79,0.0,1.75,0.00,2025,7,USD,Miles,False
1,1,2025-07-01 00:23:28,2025-07-01 01:07:44,1.0,17.70,2.0,N,132,142,1,...,1.0,80.75,2.5,1.75,0.00,2025,7,USD,Miles,False
2,2,2025-07-01 00:53:50,2025-07-01 01:27:12,1.0,9.98,1.0,N,138,48,1,...,1.0,66.97,2.5,1.75,0.75,2025,7,USD,Miles,False
3,2,2025-07-01 00:58:49,2025-07-01 01:15:55,1.0,10.27,1.0,N,138,229,1,...,1.0,72.24,2.5,1.75,0.75,2025,7,USD,Miles,False
4,2,2025-07-01 00:09:22,2025-07-01 00:23:54,1.0,2.94,1.0,N,211,97,1,...,1.0,25.75,2.5,0.00,0.75,2025,7,USD,Miles,False


In [90]:
df.count()

VendorID                 3898963
tpep_pickup_datetime     3898963
tpep_dropoff_datetime    3898963
passenger_count          2860208
trip_distance            3898963
RatecodeID               2860208
store_and_fwd_flag       2860208
PULocationID             3898963
DOLocationID             3898963
payment_type             3898963
fare_amount              3898963
extra                    3898963
mta_tax                  3898963
tip_amount               3898963
tolls_amount             3898963
improvement_surcharge    3898963
total_amount             3898963
congestion_surcharge     2860208
Airport_fee              2860208
cbd_congestion_fee       3898963
year                     3898963
month                    3898963
currency                 3898963
trip_distance_unit       3898963
suspicious               3898963
dtype: int64

In [91]:
print(df.isna().sum())

VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1038755
trip_distance                  0
RatecodeID               1038755
store_and_fwd_flag       1038755
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1038755
Airport_fee              1038755
cbd_congestion_fee             0
year                           0
month                          0
currency                       0
trip_distance_unit             0
suspicious                     0
dtype: int64


In [92]:
print(df.dtypes)

VendorID                          int64
tpep_pickup_datetime             object
tpep_dropoff_datetime            object
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag       string[python]
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
year                              int64
month                             int64
currency                         object
trip_distance_unit               object
suspicious                         bool


In [93]:
df.describe()

,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,year,month
count,3.898963e+06,2.860208e+06,3.898963e+06,2.860208e+06,3.898963e+06,3.898963e+06,3.898963e+06,3.898963e+06,3.898963e+06,3.898963e+06,3.898963e+06,3.898963e+06,3.898963e+06,3.898963e+06,2.860208e+06,2.860208e+06,3.898963e+06,3898963.0,3898963.0
mean,1.887506e+00,1.315910e+00,7.104756e+00,3.005781e+00,1.593296e+02,1.590559e+02,9.270137e-01,1.854769e+01,1.107153e+00,4.758367e-01,2.686127e+00,5.004544e-01,9.459860e-01,2.682744e+01,2.145727e+00,1.587607e-01,5.356467e-01,2025.0,7.0
std,7.337028e-01,7.589974e-01,6.700148e+02,1.349034e+01,6.624882e+01,7.062472e+01,7.939307e-01,2.007138e+01,1.803865e+00,2.659264e+00,3.935143e+00,2.132647e+00,3.001838e-01,2.427461e+01,1.008222e+00,5.558867e-01,3.599308e-01,0.0,0.0
min,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,-1.591300e+03,-7.500000e+00,-5.000000e-01,-5.275000e+01,-1.128400e+02,-1.000000e+00,-1.634750e+03,-2.500000e+00,-1.750000e+00,-7.500000e-01,2025.0,7.0
25%,2.000000e+00,1.000000e+00,1.060000e+00,1.000000e+00,1.140000e+02,1.070000e+02,0.000000e+00,8.820000e+00,0.000000e+00,5.000000e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.580000e+01,2.500000e+00,0.000000e+00,0.000000e+00,2025.0,7.0
50%,2.000000e+00,1.000000e+00,1.910000e+00,1.000000e+00,1.610000e+02,1.610000e+02,1.000000e+00,1.420000e+01,0.000000e+00,5.000000e-01,2.000000e+00,0.000000e+00,1.000000e+00,2.142000e+01,2.500000e+00,0.000000e+00,7.500000e-01,2025.0,7.0
75%,2.000000e+00,1.000000e+00,3.900000e+00,1.000000e+00,2.300000e+02,2.310000e+02,1.000000e+00,2.330000e+01,2.500000e+00,5.000000e-01,3.850000e+00,0.000000e+00,1.000000e+00,3.078000e+01,2.500000e+00,0.000000e+00,7.500000e-01,2025.0,7.0
max,7.000000e+00,9.000000e+00,3.979944e+05,9.900000e+01,2.650000e+02,2.650000e+02,4.000000e+00,2.495000e+03,1.500000e+01,5.243380e+03,5.000000e+02,2.041800e+02,1.000000e+00,5.297870e+03,2.500000e+00,6.750000e+00,1.500000e+00,2025.0,7.0


In [94]:
df.describe().style.format("{:.2f}")

,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,year,month
count,3898963.00,2860208.00,3898963.00,2860208.00,3898963.00,3898963.00,3898963.00,3898963.00,3898963.00,3898963.00,3898963.00,3898963.00,3898963.00,3898963.00,2860208.00,2860208.00,3898963.00,3898963.00,3898963.00
mean,1.89,1.32,7.10,3.01,159.33,159.06,0.93,18.55,1.11,0.48,2.69,0.50,0.95,26.83,2.15,0.16,0.54,2025.00,7.00
std,0.73,0.76,670.01,13.49,66.25,70.62,0.79,20.07,1.80,2.66,3.94,2.13,0.30,24.27,1.01,0.56,0.36,0.00,0.00
min,1.00,0.00,0.00,1.00,1.00,1.00,0.00,-1591.30,-7.50,-0.50,-52.75,-112.84,-1.00,-1634.75,-2.50,-1.75,-0.75,2025.00,7.00
25%,2.00,1.00,1.06,1.00,114.00,107.00,0.00,8.82,0.00,0.50,0.00,0.00,1.00,15.80,2.50,0.00,0.00,2025.00,7.00
50%,2.00,1.00,1.91,1.00,161.00,161.00,1.00,14.20,0.00,0.50,2.00,0.00,1.00,21.42,2.50,0.00,0.75,2025.00,7.00
75%,2.00,1.00,3.90,1.00,230.00,231.00,1.00,23.30,2.50,0.50,3.85,0.00,1.00,30.78,2.50,0.00,0.75,2025.00,7.00
max,7.00,9.00,397994.37,99.00,265.00,265.00,4.00,2495.00,15.00,5243.38,500.00,204.18,1.00,5297.87,2.50,6.75,1.50,2025.00,7.00


In [95]:
print(df[df["trip_distance"] < 0].shape[0])
print(df[df["trip_distance"] > 100].shape[0])

0
274


In [96]:
df["VendorID"].unique() 

array([1, 2, 7, 6])

In [97]:
df["passenger_count"].unique() 

array([ 1.,  2.,  3.,  0.,  4.,  5.,  6.,  8.,  9.,  7., nan])

In [98]:
df["RatecodeID"].unique() 

array([ 1.,  2.,  4.,  5., 99.,  3.,  6., nan])

In [99]:
df["store_and_fwd_flag"].unique()

<StringArray>
['N', 'Y', <NA>]
Length: 3, dtype: string

In [100]:
df["payment_type"].unique() 

array([1, 3, 2, 4, 0])

In [101]:
df["month"].unique()

array([7])

In [102]:
print(df[df["fare_amount"] < 0].shape[0]) 

247088


In [103]:
print(df[df["tip_amount"] < 0].shape[0]) 

122


In [104]:
if not is_empty(df): 
    df_normalised = apply_master_schema(df = df, logger = logger)

In [105]:
df_normalised.head()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_forward_flag,pickup_location_id,dropoff_location_id,payment_type_id,...,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,year,month,currency,trip_distance_unit,suspicious
0,1,2025-07-01 00:29:37,2025-07-01 00:45:30,1,7.3,1,N,138,74,1,...,1.0,54.790001,0.0,1.75,0.0,2025,7,USD,Miles,False
1,1,2025-07-01 00:23:28,2025-07-01 01:07:44,1,17.700001,2,N,132,142,1,...,1.0,80.75,2.5,1.75,0.0,2025,7,USD,Miles,False
2,2,2025-07-01 00:53:50,2025-07-01 01:27:12,1,9.98,1,N,138,48,1,...,1.0,66.970001,2.5,1.75,0.75,2025,7,USD,Miles,False
3,2,2025-07-01 00:58:49,2025-07-01 01:15:55,1,10.27,1,N,138,229,1,...,1.0,72.239998,2.5,1.75,0.75,2025,7,USD,Miles,False
4,2,2025-07-01 00:09:22,2025-07-01 00:23:54,1,2.94,1,N,211,97,1,...,1.0,25.75,2.5,0.0,0.75,2025,7,USD,Miles,False


In [106]:
print(df_normalised.isna().sum())

vendor_id                       0
pickup_datetime                 0
dropoff_datetime                0
passenger_count           1038755
trip_distance                   0
rate_code_id              1038755
store_and_forward_flag    1038755
pickup_location_id              0
dropoff_location_id             0
payment_type_id                 0
fare_amount                     0
extra_charge                    0
metered_rate_tax                0
tip_amount                      0
tolls_amount                    0
improvement_surcharge           0
total_amount                    0
congestion_surcharge      1038755
airport_fee               1038755
cbd_congestion_fee              0
year                            0
month                           0
currency                        0
trip_distance_unit              0
suspicious                      0
dtype: int64


In [107]:
df_normalised.dtypes

vendor_id                          UInt8
pickup_datetime           datetime64[ns]
dropoff_datetime          datetime64[ns]
passenger_count                    UInt8
trip_distance                    Float32
rate_code_id                       UInt8
store_and_forward_flag          category
pickup_location_id                 Int16
dropoff_location_id                Int16
payment_type_id                    UInt8
fare_amount                      Float32
extra_charge                     Float32
metered_rate_tax                 Float32
tip_amount                       Float32
tolls_amount                     Float32
improvement_surcharge            Float32
total_amount                     Float32
congestion_surcharge             Float32
airport_fee                      Float32
cbd_congestion_fee               Float32
year                               Int16
month                              UInt8
currency                        category
trip_distance_unit              category
suspicious      

In [109]:
if not is_empty(df_normalised):
    slice_path = Path("../data/CSV/removed/1st-slice/")
    slice_path.mkdir(parents = True, exist_ok = True)
    df_cleaned = cleaning_transformation(df = df_normalised, removed_slice_path = slice_path, logger = logger, process_type = IN_MEMORY, header = True)

In [115]:
df_cleaned[df_cleaned["tip_amount"] == -52.75].T

,2388196
vendor_id,2
pickup_datetime,2025-07-27 03:26:03
dropoff_datetime,2025-07-27 03:28:34
passenger_count,1
trip_distance,0.17
rate_code_id,1
store_and_forward_flag,N
pickup_location_id,158
dropoff_location_id,249
payment_type_id,4


In [116]:
summary = pd.DataFrame()
if not is_empty(df_cleaned):
    summary = profile_vendor_specific_differences(df_cleaned)
    
summary.head()

,trip_distance_count,trip_distance_mean,trip_distance_median,trip_distance_std,trip_distance_min,trip_distance_max,fare_amount_count,fare_amount_mean,fare_amount_median,fare_amount_std,...,tip_amount_median,tip_amount_std,tip_amount_min,tip_amount_max,total_amount_count,total_amount_mean,total_amount_median,total_amount_std,total_amount_min,total_amount_max
vendor_id,,,,,,,,,,,,,,,,,,,,,
1,724020,3.503339,1.9,4.304075,0.0,363.100006,724020,19.61042,14.2,16.585106,...,2.0,3.739249,0.0,500.0,724020,27.608032,21.709999,19.838491,0.0,1098.060059
2,3115682,3.535274,1.93,4.456337,0.0,366.399994,3115682,18.345434,14.2,20.851797,...,1.95,3.986356,-52.75,400.5,3115682,26.657742,21.370001,25.267557,-1634.75,5297.870117
6,2302,8.462172,7.495,5.571377,0.26,29.85,2302,2.944092,2.9,1.166767,...,0.0,0.0,0.0,0.0,2302,30.856983,27.904999,15.286493,0.0,99.739998


In [117]:
missing_values = pd.Series(dtype="int64")
if not is_empty(df_cleaned):
    missing_values = profile_missing_values(df_cleaned)

print(missing_values)

                        missing_count  missing_percent
rate_code_id                  1038447            27.03
congestion_surcharge          1038447            27.03
airport_fee                   1038447            27.03
vendor_id                           0             0.00
pickup_datetime                     0             0.00
trip_distance                       0             0.00
store_and_forward_flag              0             0.00
passenger_count                     0             0.00
dropoff_datetime                    0             0.00
dropoff_location_id                 0             0.00
pickup_location_id                  0             0.00
payment_type_id                     0             0.00
fare_amount                         0             0.00
tip_amount                          0             0.00
tolls_amount                        0             0.00
extra_charge                        0             0.00
metered_rate_tax                    0             0.00
total_amou

##### Benchmark adding cleaning and transformation, all the data in memory

In [42]:
# Build benchmark return for ingestion step
def build_return_dict_cleaning_transformation(
    slice_path: str | Path,
    operation: str,
    ntest: int, 
    processed_files: int,
    processed_rows: int,
    rows_after_cleaning: int,
    execution_time: float,
    throughput: float,
    benchmark_start_timestamp: str,
    benchmark_end_timestamp: str,
    msg: str,
    summary: pd.DataFrame,
    missing_values: pd.DataFrame,
    chunksize: int = 0,
    resource_stats: dict[str, Any] | None = None,
) -> dict[str, Any]:
    
    if resource_stats is None:
        resource_stats = ResourceMonitor.as_dict()

    return {
        # operations:
        "stage": CLEANING_TRANSFORMATION,
        # Type of operation
        "operation": operation,
        # Test Number
        "test_number": ntest,   
        # Slice path
        "slice_path": str(slice_path),
        # Performance
        "processed_files": processed_files,
        "processed_rows": processed_rows,
        "rows_after_cleaning": rows_after_cleaning,
        "execution_time_sec": execution_time,
        "throughput_rows_sec": throughput,
        "chunksize": chunksize,
        "dataset_size_mib": compute_dataset_size_mib(slice_path),
        # Timestamps
        "benchmark_start_timestamp": benchmark_start_timestamp,
        "benchmark_end_timestamp": benchmark_end_timestamp,
        # Resources
        "avg_rss_mib": resource_stats["avg_rss_mib"],
        "peak_rss_mib": resource_stats["peak_rss_mib"],
        "memory_percent_of_total": resource_stats["memory_percent_of_total"],
        "avg_process_cpu_percent": resource_stats["avg_process_cpu_percent"],
        "peak_process_cpu_percent": resource_stats["peak_process_cpu_percent"],
        "avg_system_cpu_per_core_percent": resource_stats["avg_system_cpu_per_core_percent"],
        "peak_system_cpu_per_core_percent": resource_stats["peak_system_cpu_per_core_percent"],
        "read_mib": resource_stats["read_mib"],
        "write_mib": resource_stats["write_mib"],    
        # Message
        "msg": msg,
        # Profiling
        "summary": summary,
        "missing_values": missing_values
    } 

In [43]:
# Benchmark cleaning, transformation and profiling using Pandas library. Ingestion
# is performed before the benchmarking
# It also records execution time, throughput, timestamps,
# resource usage statistics
# Parameters:
# slice_path: input slice directory
# ntest: test number
# logger: Logger instance for reporting progress and errors
# Returns:
# benchmark results including slice path, processed files, processed rows, rows after cleaning, 
# execution time, throughput, timestamps, resource statistics, a message and profiling results
def benchmark_pandas_cleaning_transformation(
    slice_path: str | Path,
    ntest: int,
    logger: logging.Logger | None = None
) -> dict[str, Any]:

    if logger is None:
        logger = setup_logger()

    timer = Timer()
    monitor = ResourceMonitor()
    benchmark_start_timestamp = Timer.get_timestamp()
    timer.start("total")
    monitor.start()
    MSG_ERROR = "Ingestion, cleaning, transformation and profiling completed with errors, check pipeline.log"
    REMOVED_FOLDER = "removed"

    df, total_files, total_rows, ingested = pandas_ingestion(slice_path = slice_path, logger = logger)
    if not ingested:
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        throughput = total_rows / execution_time if execution_time > 0 else 0.0
        return build_return_dict_cleaning_transformation(slice_path = slice_path, operation = IN_MEMORY, ntest = ntest, processed_files = total_files, processed_rows = total_rows,
                                                         rows_after_cleaning = 0, execution_time = execution_time, throughput = throughput,
                                                         benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                                         resource_stats = resource_stats, msg = MSG_ERROR, 
                                                         summary = pd.DataFrame(), missing_values = pd.DataFrame())
    
    df_normalised = pd.DataFrame()
    if not is_empty(df): 
        # Apply new schema
        df_normalised = apply_master_schema(df = df, logger = logger)
    df_cleaned = pd.DataFrame()
    if not is_empty(df_normalised):
        # Clean dataset and transform it
        removed_slice_path = Path(slice_path)
        removed_slice_path = removed_slice_path.parents[1] / REMOVED_FOLDER / removed_slice_path.name
        removed_slice_path.mkdir(parents = True, exist_ok = True)
        df_cleaned = cleaning_transformation(df = df_normalised, removed_slice_path = removed_slice_path, logger = logger, process_type = IN_MEMORY, header = True)
    # Profiling
    summary = pd.DataFrame()
    missing_values = pd.DataFrame()
    if not is_empty(df_cleaned):
        summary = profile_vendor_specific_differences(df_cleaned)
        missing_values = profile_missing_values(df_cleaned)
    
    timer.stop("total")
    resource_stats = monitor.stop()
    benchmark_end_timestamp = Timer.get_timestamp()
    
    execution_time = timer.duration("total")
    throughput = total_rows / execution_time if execution_time > 0 else 0

    print(f"Pandas cleaned and transformed {total_files} file(s) and {total_rows} row(s) in {execution_time:.2f} seconds")
    print(f"Throughput: {throughput:.2f} rows/sec")

    return build_return_dict_cleaning_transformation(slice_path = slice_path, operation = IN_MEMORY, ntest = ntest, processed_files = total_files, processed_rows = total_rows,
                                                     rows_after_cleaning = len(df_cleaned) if not is_empty(df_cleaned) else 0, 
                                                     execution_time = execution_time, throughput = throughput,
                                                     benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                                     resource_stats = resource_stats, msg = "Ingestion, cleaning, transformation and profiling completed successfully", 
                                                     summary = summary, missing_values = missing_values)  
    

In [44]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()

result = benchmark_pandas_cleaning_transformation(slice_path = "../data/CSV/raw-data/1st-slice/", logger = logger, ntest = 1)
log_info_msg(logger, result["msg"])

Reading ../data/CSV/raw-data/1st-slice/yellow_tripdata_2025-07.csv...
Pandas cleaned and transformed 1 file(s) and 3898963 row(s) in 205.93 seconds
Throughput: 18933.06 rows/sec


In [45]:
print(result)

{'stage': 'ingestion, cleaning, transformation and profiling', 'operation': 'in-memory', 'test_number': 1, 'slice_path': '../data/CSV/raw-data/1st-slice/', 'processed_files': 1, 'processed_rows': 3898963, 'rows_after_cleaning': 3842004, 'execution_time_sec': 205.934143318, 'throughput_rows_sec': 18933.05761337151, 'chunksize': 0, 'dataset_size_mib': 402.3603820800781, 'benchmark_start_timestamp': '2026-04-12T09:58:34.371225+01:00', 'benchmark_end_timestamp': '2026-04-12T10:02:00.340276+01:00', 'avg_rss_mib': 1801.0911247834872, 'peak_rss_mib': 3948.58984375, 'memory_percent_of_total': 50.32815599334029, 'avg_process_cpu_percent': 127.38290993071601, 'peak_process_cpu_percent': 166.5, 'avg_system_cpu_per_core_percent': [25.267205542725172, 20.77090069284065, 39.44018475750576, 36.09284064665128], 'peak_system_cpu_per_core_percent': [100.0, 100.0, 100.0, 100.0], 'read_mib': 406.796875, 'write_mib': 7.10546875, 'msg': 'Ingestion, cleaning, transformation and profiling completed successful

##### Benchmark adding cleaning and transformation using chunks

In [44]:
# Benchmark chunked ingestion, cleaning, transformation and
# profiling using Pandas library. It also records execution time, throughput, timestamps,
# resource usage statistics
# Parameters:
# slice_path: input slice directory
# logger: Logger instance for reporting progress and errors
# ntest: test number
# Returns:
# benchmark results including slice path, processed files, processed rows, rows after cleaning, execution time, 
# throughput, timestamps, resource statistics, a message and profiling results
def benchmark_pandas_cleaning_transformation_chunk(
    slice_path: str | Path,
    ntest: int,
    chunksize: int = 100_000, 
    logger: logging.Logger | None = None
) -> dict[str, Any]: 

    if logger is None:
        logger = setup_logger()

    timer = Timer()
    monitor = ResourceMonitor()
    benchmark_start_timestamp = Timer.get_timestamp()
    timer.start("total")
    monitor.start()

    log_info_msg(logger, "NYC Yellow trip taxi dataset")
    MSG_ERROR = "Ingestion, cleaning, transformation and profiling completed with errors, check pipeline.log"
    REMOVED_FOLDER = "removed"
    CURRENCY = "USD"
    DISTANCE_UNIT = "Miles"
    processed_rows = 0
    
    slice_path = Path(slice_path)
    files = sorted(slice_path.glob(CSV_PATTERN))

    if len(files) == 0:
        log_error_msg(logger, f"No CSV files found in folder: {slice_path}")
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        throughput = processed_rows / execution_time if execution_time > 0 else 0.0
        return build_return_dict_cleaning_transformation(slice_path = slice_path, operation = CHUNKING, ntest = ntest, processed_files = 0, processed_rows = processed_rows,
                                                         rows_after_cleaning = 0, execution_time = execution_time, throughput = throughput,
                                                         benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                                         resource_stats = resource_stats, msg = MSG_ERROR, summary = pd.DataFrame(),
                                                         chunksize = chunksize, missing_values = pd.DataFrame())    
    processed_files = 0
    rows_after_cleaning = 0
    first_chunk_header = True
    missing_counts_total = None
    missing_values = pd.DataFrame()

    for f in files:
        print(f"Reading {f}...")

        year, month, msg = extract_year_month_from_filename(f)

        if year is None or month is None:
            log_error_msg(logger, msg)
            timer.stop("total")
            resource_stats = monitor.stop()
            benchmark_end_timestamp = Timer.get_timestamp()
            execution_time = timer.duration("total")
            throughput = processed_rows / execution_time if execution_time > 0 else 0.0

            return build_return_dict_cleaning_transformation(slice_path = slice_path, operation = CHUNKING, ntest = ntest, processed_files = processed_files, processed_rows = processed_rows,
                                                             rows_after_cleaning = 0, execution_time = execution_time, throughput = throughput,
                                                             benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                                             resource_stats = resource_stats, msg = MSG_ERROR, summary = pd.DataFrame(), 
                                                             chunksize = chunksize, missing_values = pd.DataFrame())    
        
        try: 
            chunk_iter = pd.read_csv(
                f,
                chunksize = chunksize,
                dtype = {"store_and_fwd_flag": "string"},
                low_memory = False
            )
        except Exception as e:
            log_error_msg(logger, f"Failed to open file {f} as chunked CSV: {type(e).__name__}: {e}")
            timer.stop("total")
            resource_stats = monitor.stop()
            benchmark_end_timestamp = Timer.get_timestamp()
            execution_time = timer.duration("total")
            throughput = processed_rows / execution_time if execution_time > 0 else 0.0
            return build_return_dict_cleaning_transformation(slice_path = slice_path, operation = CHUNKING, ntest = ntest, processed_files = processed_files, processed_rows = processed_rows,
                                                             rows_after_cleaning = rows_after_cleaning, execution_time = execution_time, throughput = throughput,
                                                             benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                                             resource_stats = resource_stats, msg = MSG_ERROR, summary = pd.DataFrame(), 
                                                             chunksize = chunksize, missing_values = pd.DataFrame())    
            
        path = Path(f)
        removed_slice_path = path.parents[2] / REMOVED_FOLDER / path.parent.name
        removed_slice_path.mkdir(parents = True, exist_ok = True)
        for chunk in chunk_iter:
            chunk["year"] = year
            chunk["month"] = month
            chunk["currency"] = CURRENCY
            chunk["trip_distance_unit"] = DISTANCE_UNIT
            chunk["suspicious"] = False
            df_normalised = pd.DataFrame()
            if not is_empty(chunk): 
                # Apply new schema
                df_normalised = apply_master_schema(chunk, logger)
            df_cleaned = pd.DataFrame()
            if not is_empty(df_normalised):
                # Clean dataset and transform it
                df_cleaned = cleaning_transformation(df = df_normalised, removed_slice_path = removed_slice_path, logger = logger, process_type = CHUNKING, header = first_chunk_header)
                first_chunk_header = False
            rows_after_cleaning += len(df_cleaned)
            # Profiling
            if not is_empty(df_cleaned):
                missing_counts_total = update_profile_missing_values(df_cleaned, missing_counts_total)
            processed_rows += len(chunk)

            # Cleanup
            del chunk
            del df_normalised
            del df_cleaned
            
        processed_files += 1
        del chunk_iter
        gc.collect()

    missing_values = finalise_profile_missing_values(missing_counts_total, rows_after_cleaning)
    
    timer.stop("total")
    resource_stats = monitor.stop()
    benchmark_end_timestamp = Timer.get_timestamp()
    
    execution_time = timer.duration("total")
    throughput = processed_rows / execution_time if execution_time > 0 else 0
    
    print(f"Pandas cleaned and transformed {processed_files} file(s) and {processed_rows} row(s) in {execution_time:.2f} seconds")
    print(f"Throughput: {throughput:.2f} rows/sec")

    return build_return_dict_cleaning_transformation(slice_path = slice_path, operation = CHUNKING, ntest = ntest, processed_files = processed_files, processed_rows = processed_rows,
                                                     rows_after_cleaning = rows_after_cleaning, execution_time = execution_time, throughput = throughput,
                                                     benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                                     resource_stats = resource_stats, msg = "Ingestion, cleaning, transformation and profiling completed successfully. Vendor summary is not computed in chunk mode.", 
                                                     summary = pd.DataFrame(), 
                                                     chunksize = chunksize, missing_values = missing_values)    
             

In [195]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
result = benchmark_pandas_cleaning_transformation_chunk(slice_path = "../data/CSV/raw-data/2nd-slice/", logger = logger, ntest = 1)
log_info_msg(logger, result["msg"])

Reading ../data/CSV/raw-data/2nd-slice/yellow_tripdata_2025-05.csv...
Reading ../data/CSV/raw-data/2nd-slice/yellow_tripdata_2025-06.csv...
Reading ../data/CSV/raw-data/2nd-slice/yellow_tripdata_2025-07.csv...
Pandas cleaned and transformed 3 file(s) and 12813768 row(s) in 488.52 seconds
Throughput: 26229.85 rows/sec


In [196]:
print(result)

{'stage': 'ingestion, cleaning, transformation and profiling', 'operation': 'chunking', 'test_number': 1, 'slice_path': '../data/CSV/raw-data/2nd-slice', 'processed_files': 3, 'processed_rows': 12813768, 'rows_after_cleaning': 12621491, 'execution_time_sec': 488.5185617090028, 'throughput_rows_sec': 26229.848780306555, 'chunksize': 100000, 'dataset_size_mib': 1321.7953338623047, 'benchmark_start_timestamp': '2026-04-10T14:32:47.907091+01:00', 'benchmark_end_timestamp': '2026-04-10T14:40:56.575298+01:00', 'avg_rss_mib': 558.6876311800373, 'peak_rss_mib': 865.25390625, 'memory_percent_of_total': 11.028401351060694, 'avg_process_cpu_percent': 123.06119402985081, 'peak_process_cpu_percent': 164.7, 'avg_system_cpu_per_core_percent': [30.13706467661693, 36.239303482587054, 23.908955223880618, 28.967910447761177], 'peak_system_cpu_per_core_percent': [100.0, 100.0, 100.0, 100.0], 'read_mib': 1275.515625, 'write_mib': 25.44921875, 'msg': 'Ingestion, cleaning, transformation and profiling comple

In [197]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
result = benchmark_pandas_cleaning_transformation_chunk(slice_path = "../data/CSV/raw-data/3rd-slice/", logger = logger, ntest = 1)
log_info_msg(logger, result["msg"])

Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-06.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-07.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-08.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-09.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-10.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-11.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-12.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-01.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-02.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-03.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-04.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-05.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-06.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-07.csv...
Pandas cleaned and t

In [198]:
print(result)

{'stage': 'ingestion, cleaning, transformation and profiling', 'operation': 'chunking', 'test_number': 1, 'slice_path': '../data/CSV/raw-data/3rd-slice', 'processed_files': 14, 'processed_rows': 52359167, 'rows_after_cleaning': 52083818, 'execution_time_sec': 1630.552163698001, 'throughput_rows_sec': 32111.310613487112, 'chunksize': 100000, 'dataset_size_mib': 5343.146864891052, 'benchmark_start_timestamp': '2026-04-10T14:45:24.950590+01:00', 'benchmark_end_timestamp': '2026-04-10T15:12:35.636521+01:00', 'avg_rss_mib': 387.3974578152753, 'peak_rss_mib': 400.921875, 'memory_percent_of_total': 5.110092327791541, 'avg_process_cpu_percent': 121.28621669626973, 'peak_process_cpu_percent': 160.5, 'avg_system_cpu_per_core_percent': [32.73275310834814, 33.938436944937855, 26.643126110124356, 28.96916518650098], 'peak_system_cpu_per_core_percent': [100.0, 100.0, 100.0, 100.0], 'read_mib': 3449.3515625, 'write_mib': 41.8046875, 'msg': 'Ingestion, cleaning, transformation and profiling completed 

In [199]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
result = benchmark_pandas_cleaning_transformation_chunk(slice_path = "../data/CSV/raw-data/4th-slice/", logger = logger, ntest = 1)
log_info_msg(logger, result["msg"])

Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-01.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-02.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-03.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-04.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-05.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-06.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-07.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-08.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-09.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-10.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-11.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-12.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2024-01.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2024-02.csv...
Reading ../data/CSV/

In [200]:
print(result)

{'stage': 'ingestion, cleaning, transformation and profiling', 'operation': 'chunking', 'test_number': 1, 'slice_path': '../data/CSV/raw-data/4th-slice', 'processed_files': 31, 'processed_rows': 107462293, 'rows_after_cleaning': 107143226, 'execution_time_sec': 3910.629317117, 'throughput_rows_sec': 27479.53955380857, 'chunksize': 100000, 'dataset_size_mib': 10898.749529838562, 'benchmark_start_timestamp': '2026-04-10T15:14:23.019463+01:00', 'benchmark_end_timestamp': '2026-04-10T16:19:33.796989+01:00', 'avg_rss_mib': 326.5348504770059, 'peak_rss_mib': 399.1953125, 'memory_percent_of_total': 5.0880858114728635, 'avg_process_cpu_percent': 123.05839229263893, 'peak_process_cpu_percent': 237.7, 'avg_system_cpu_per_core_percent': [30.20239349691398, 36.03697124792998, 23.984239048622573, 28.15837723919911], 'peak_system_cpu_per_core_percent': [100.0, 100.0, 100.0, 100.0], 'read_mib': 10890.23828125, 'write_mib': 55.84765625, 'msg': 'Ingestion, cleaning, transformation and profiling complet

#### Persist results to parquet Stage

##### Benchmark adding persistence to parquet, all the data in memory

In [45]:
import duckdb

In [46]:
# Perform in-memory cleaning, transformation and profiling using Pandas library
# Ingest all CSV files from the given slice directory
# Applies schema normalisation
# Performs cleaning and transformation steps
# Computes profiling statistics
# Parameters:
# slice_path: input slice directory containing CSV files
# Returns:
# cleander and transformed DataFrame, number of successfully processed files,
# if number of successfully processed files, profiling DataFrames 
def pandas_cleaning_transformation(
    slice_path: str | Path, 
    logger: logging.Logger | None = None
) -> dict[pd.DataFrame, int, int, bool, pd.DataFrame, pd.DataFrame]:

    if logger is None:
        logger = setup_logger()

    REMOVED_FOLDER = "removed"
    
    df, total_files, total_rows, ingested = pandas_ingestion(slice_path = slice_path, logger = logger)

    if not ingested:
        return df, total_files, total_rows, False, pd.DataFrame, pd.DataFrame
    
    df_normalised = pd.DataFrame()
    if not is_empty(df): 
        # Apply new schema
        df_normalised = apply_master_schema(df = df, logger = logger)
    df_cleaned = pd.DataFrame()
    if not is_empty(df_normalised):
        # Clean dataset and transform it
        removed_slice_path = Path(slice_path)
        removed_slice_path = removed_slice_path.parents[1] / REMOVED_FOLDER / removed_slice_path.name
        removed_slice_path.mkdir(parents = True, exist_ok = True)
        df_cleaned = cleaning_transformation(df = df_normalised, removed_slice_path = removed_slice_path, logger = logger, process_type = IN_MEMORY, header = True)
    # Profiling
    summary = pd.DataFrame()
    missing_values = pd.DataFrame()
    if not is_empty(df_cleaned):
        summary = profile_vendor_specific_differences(df_cleaned)
        missing_values = profile_missing_values(df_cleaned)

    log_info_msg(logger, "Cleaning and transformation completed")
    
    return df_cleaned, total_files, total_rows, True, summary, missing_values

In [47]:
# Persist a DataFrame to partitioned parquet files using DuckDB
# Data is partitioned by year and month
# parquet_path/year=YYYY/month=MM/
# Parameters:
# df: input DataFrame to persist
# parquet_path: output directory for parquet files
# table_name: temporary DuckDB table name used for the DataFrame
# threads: number of DuckDB execution threads
# logger: Logger instance for reporting progress and errors
# verify_count_rows: if to validate written parquet files by counting rows
# Returns:
# True if persistence completed successfully, false otherwise
# The number of created files
# Total size of the parquet files
def persist_partitioned_parquet_duckdb(
    df: pd.DataFrame, 
    parquet_path: str | Path, 
    table_name: str = "yellow_taxi", 
    threads: int = 0, 
    logger: logging.Logger | None = None, 
    verify_count_rows: bool = False,
    connection: duckdb.DuckDBPyConnection | None = None
) -> tuple[bool, int, float]:

    if logger is None:
        logger = setup_logger()

    MIB = 1024 * 1024
    valid_mask = df["year"].notna() & df["month"].notna()
    expected_rows = int(valid_mask.sum())
    nrows = len(df)
    
    if expected_rows != nrows:
        log_error_msg(logger, "Some rows have null year/month values, parquet persistence aborted")
        return False, 0, 0.0

    created_files = set()
    parquet_files_size_mib = 0.0
    created_local_connection = False
    
    try: 
        if connection is None:
            connection = duckdb.connect()
            created_local_connection = True
        if threads and threads > 0:
            connection.execute(f"PRAGMA threads={threads}")
    except duckdb.Error as e:
        msg = f"Failed to initialize DuckDB connection (threads = {threads}): {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        return False, len(created_files), parquet_files_size_mib          

    try:
        
        connection.register(table_name, df)

        parquet_path = Path(parquet_path)
    
        existing_files = set(parquet_path.rglob("*.parquet"))
        
        query = f"""
        COPY (
            SELECT *
            FROM {table_name}
            WHERE year IS NOT NULL AND month IS NOT NULL
        ) TO '{parquet_path.as_posix()}'
        (FORMAT PARQUET, PARTITION_BY (year, month), APPEND true, FILENAME_PATTERN 'data_{{uuid}}');
        """

        connection.execute(query)
        
        new_files = set(parquet_path.rglob("*.parquet"))

        created_files = new_files - existing_files
        
        if not created_files:
            msg = f"No new parquet files were created in: {parquet_path}"
            log_error_msg(logger, msg)
            return False, len(created_files), parquet_files_size_mib

        size_bytes = [f.stat().st_size for f in created_files]
        parquet_files_size_mib = sum(size_bytes) / MIB

        created_paths = [f.as_posix() for f in created_files]
        paths_sql = ", ".join(f"'{p}'" for p in created_paths)
        
        if verify_count_rows:
            readback_nrows = connection.execute(f"SELECT COUNT(*) FROM read_parquet([{paths_sql}])").fetchone()[0]
            if readback_nrows != expected_rows:
                msg = f"Expected {expected_rows} rows, but new parquet files contain {readback_nrows}"
                log_error_msg(logger, msg)
                return False, len(created_files), parquet_files_size_mib

        msg = f"DataFrame persisted successfully to {parquet_path}. Created {len(created_files)} parquet file(s) with {expected_rows} row(s)"
        log_info_msg(logger, msg)
        return True, len(created_files), parquet_files_size_mib
               
    except Exception as e:
        msg = f"Failed to persist partitioned parquet: {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        return False, len(created_files), parquet_files_size_mib
        
    finally: 
        if connection is not None:
            try:
                connection.unregister(table_name)
            except Exception:
                pass
        if created_local_connection and connection is not None:
            connection.close()

In [48]:
# Build benchmark return for parquet persistence step
def build_return_dict_parquet_persistence(
    slice_path: str | Path,
    parquet_path: str | Path,
    operation: str,
    ntest: int, 
    processed_files: int,
    processed_rows: int,
    rows_after_cleaning: int,
    execution_time: float,
    throughput: float,
    parquet_files_size_mib: float,
    created_parquet_files: int,
    threads: int,
    benchmark_start_timestamp: str,
    benchmark_end_timestamp: str,
    msg: str,
    summary: pd.DataFrame,
    missing_values: pd.DataFrame,
    chunksize: int = 0,
    resource_stats: dict[str, Any] | None = None,
) -> dict[str, Any]:
 
    if resource_stats is None:
        resource_stats = ResourceMonitor.as_dict()

    return {
        # operations:
        "stage": PERSISTENCE_TO_PARQUET,
        # Type of operation
        "operation": operation,
        # Test Number
        "test_number": ntest,   
        # Slice path
        "slice_path": str(slice_path),
        # Parquet path:
        "parquet_path": str(parquet_path),
        # Performance
        "processed_files": processed_files,
        "processed_rows": processed_rows,
        "rows_after_cleaning": rows_after_cleaning,
        "execution_time_sec": execution_time,
        "throughput_rows_sec": throughput,
        "threads": threads,
        "chunksize": chunksize,
        "dataset_size_mib": compute_dataset_size_mib(slice_path),
        "parquet_files_size_mib": parquet_files_size_mib,
        "created_parquet_files": created_parquet_files,
        # Timestamps
        "benchmark_start_timestamp": benchmark_start_timestamp,
        "benchmark_end_timestamp": benchmark_end_timestamp,
        # Resources
        "avg_rss_mib": resource_stats["avg_rss_mib"],
        "peak_rss_mib": resource_stats["peak_rss_mib"],
        "memory_percent_of_total": resource_stats["memory_percent_of_total"],
        "avg_process_cpu_percent": resource_stats["avg_process_cpu_percent"],
        "peak_process_cpu_percent": resource_stats["peak_process_cpu_percent"],
        "avg_system_cpu_per_core_percent": resource_stats["avg_system_cpu_per_core_percent"],
        "peak_system_cpu_per_core_percent": resource_stats["peak_system_cpu_per_core_percent"],
        "read_mib": resource_stats["read_mib"],
        "write_mib": resource_stats["write_mib"],    
        # Message
        "msg": msg,
        # Profiling
        "summary": summary,
        "missing_values": missing_values
    } 

In [49]:
# Benchamrk persistence of cleaned data to partitioned parquet using duckdb
# Parameters:
# slice_path: input slice directory containing CSV files
# parquet_path: output directory for partitioned parquet files
# threads: number of DuckDB execution threads
# logger: Logger instance for reporting progress and errors
# ntest: test number
# verify_count_rows: if to verify row counts after compaction
# Returns:
# benchmark results including slice path, processed files, processed rows, rows after cleaning, execution time, 
# throughput, threads, timestamps, resource statistics, a message and profiling results
def benchmark_duckdb_parquet_persistence(
    slice_path: str | Path, 
    ntest: int,
    parquet_path: str | Path, 
    threads: int = 0, 
    logger: logging.Logger | None = None,
    verify_count_rows: bool = False,
) -> dict[str, Any]:

    if logger is None:
        logger = setup_logger()

    timer = Timer()
    monitor = ResourceMonitor()
    benchmark_start_timestamp = Timer.get_timestamp()
    timer.start("total")
    monitor.start()
    MSG_ERROR = "Ingestion, cleaning, transformation, profiling and persistence to parquet completed with errors, check pipeline.log"
    created_parquet_files = 0
    df, total_files, total_rows, cleaning_transformation, summary, missing_values = pandas_cleaning_transformation(slice_path = slice_path, logger = logger)
    
    if not cleaning_transformation:
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        throughput = total_rows / execution_time if execution_time > 0 else 0.0
        return build_return_dict_parquet_persistence(slice_path = slice_path, parquet_path = parquet_path, operation = IN_MEMORY, ntest = ntest, processed_files = total_files, processed_rows = total_rows, 
                                                     rows_after_cleaning = len(df), execution_time = execution_time, throughput = throughput,
                                                     threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                     benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), created_parquet_files = created_parquet_files,
                                                     msg = MSG_ERROR, summary = summary, missing_values = missing_values, 
                                                     parquet_files_size_mib = 0.0)
            
    persisted = False
    parquet_files_size_mib = 0.0
    if not is_empty(df):
        persisted, created_parquet_files, parquet_files_size_mib = persist_partitioned_parquet_duckdb(df = df, parquet_path = Path(parquet_path), 
                                                                                                      logger = logger, verify_count_rows = verify_count_rows)
        
    timer.stop("total")
    resource_stats = monitor.stop()
    benchmark_end_timestamp = Timer.get_timestamp()
    
    execution_time = timer.duration("total")
    
    throughput = total_rows / execution_time if execution_time > 0 else 0.0
        
    print(f"DuckDB persisted {total_files} file(s) and {total_rows} row(s) to Parquet in {execution_time:.2f} seconds")
    print(f"Throughput: {throughput:.2f} rows/sec")

    if persisted:
        msg = "Ingestion, cleaning, transformation, profiling and persistence to parquet completed successfully"
    else:
        msg = "Ingestion, cleaning, transformation, profiling and persistence to parquet completed with errors, check pipeline.log"

    return build_return_dict_parquet_persistence(slice_path = slice_path, parquet_path = parquet_path, operation = IN_MEMORY, ntest = ntest, processed_files = total_files, processed_rows = total_rows, 
                                                 rows_after_cleaning = len(df), execution_time = execution_time, throughput = throughput,
                                                 threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                 benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                                 resource_stats = resource_stats, created_parquet_files = created_parquet_files,
                                                 msg = msg, summary = summary, missing_values = missing_values,
                                                 parquet_files_size_mib = parquet_files_size_mib)
    

In [206]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
result = benchmark_duckdb_parquet_persistence(slice_path = "../data/CSV/raw-data/1st-slice/", parquet_path= "../data/parquet/1st-slice/", logger = logger, 
                                              ntest = 1, verify_count_rows = True, threads = 3)
log_info_msg(logger, result["msg"])

Reading ../data/CSV/raw-data/1st-slice/yellow_tripdata_2025-07.csv...
DuckDB persisted 1 file(s) and 3898963 row(s) to Parquet in 125.40 seconds
Throughput: 31091.30 rows/sec


In [207]:
print(result)

{'stage': 'ingestion, cleaning, transformation, profiling and persistence to parquet', 'operation': 'in-memory', 'test_number': 1, 'slice_path': '../data/CSV/raw-data/1st-slice/', 'parquet_path': '../data/parquet/1st-slice/', 'processed_files': 1, 'processed_rows': 3898963, 'rows_after_cleaning': 3842004, 'execution_time_sec': 125.40366529300081, 'throughput_rows_sec': 31091.300169657912, 'threads': 3, 'chunksize': 0, 'dataset_size_mib': 402.3603820800781, 'parquet_files_size_mib': 79.21061992645264, 'created_parquet_files': 1, 'benchmark_start_timestamp': '2026-04-10T16:28:27.195584+01:00', 'benchmark_end_timestamp': '2026-04-10T16:30:32.624680+01:00', 'avg_rss_mib': 2350.104025554187, 'peak_rss_mib': 3902.98046875, 'memory_percent_of_total': 49.746825485338285, 'avg_process_cpu_percent': 145.684236453202, 'peak_process_cpu_percent': 412.1, 'avg_system_cpu_per_core_percent': [36.95911330049262, 56.968965517241365, 23.657142857142873, 26.81576354679802], 'peak_system_cpu_per_core_perce

##### Number of rows and mbytes

In [51]:
files = glob.glob("../data/parquet/1st-slice/year=2025/month=7/*.parquet")

df_check = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

print(df_check.shape)

(3842011, 23)


In [52]:
rows = duckdb.sql("SELECT COUNT(*) FROM read_parquet('../data/parquet/1st-slice/') WHERE suspicious = False").fetchone()[0]
print(rows)

2515380


In [53]:
from pathlib import Path

def folder_size_mb(path):
    total = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())
    return total / (1024 * 1024)

print(folder_size_mb("../data/parquet/1st-slice/"))

79.11335182189941


In [54]:
#print(f"len(df): {len(df)}")
#print(f"df.columns.tolist(): {df.columns.tolist()}")
#print(f"df.shape: {df.shape}")

##### Benchmark adding persistence to parquet using chunks

In [50]:
# Benchmark chunked ingestion, cleaning, transformation,
# profiling and parquet persistence using Pandas library and DuckDB. It also records execution time, throughput, timestamps,
# resource usage statistics
# Parameters:
# slice_path: input slice directory containing CSV files
# parquet_path: output directory for partitioned parquet file
# chunksize: number of rows per chunk
# logger: Logger instance for reporting progress and errors
# threads: number of DuckDB execution threads
# ntest: test number
# verify_count_rows: if to verify row counts after compaction
# Returns:
# benchmark results including slice path, processed files, processed rows, rows after cleaning, execution time, 
# throughput, threads, timestamps, resource statistics, a message and profiling results
def benchmark_duckdb_parquet_persistence_chunk(
    slice_path: str | Path,
    ntest: int,
    parquet_path: str | Path, 
    chunksize: int = 100_000, 
    threads: int = 0, 
    logger: logging.Logger | None = None,
    verify_count_rows: bool = False,
) -> dict[str, Any]:

    if logger is None:
        logger = setup_logger()
    
    timer = Timer()
    monitor = ResourceMonitor()
    benchmark_start_timestamp = Timer.get_timestamp()
    timer.start("total")
    monitor.start()

    log_info_msg(logger, "NYC Yellow trip taxi dataset")
    MSG_ERROR = "Ingestion, cleaning, transformation, profiling and persistence to parquet completed with errors, check pipeline.log"
    REMOVED_FOLDER = "removed"
    CURRENCY = "USD"
    DISTANCE_UNIT = "Miles"
    processed_rows = 0
    slice_path = Path(slice_path)
    files = sorted(slice_path.glob(CSV_PATTERN))

    if len(files) == 0:
        log_error_msg(logger, f"No CSV files found in folder: {slice_path}")
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        throughput = processed_rows / execution_time if execution_time > 0 else 0.0
        return build_return_dict_parquet_persistence(slice_path = slice_path, parquet_path = parquet_path, operation = CHUNKING, ntest = ntest, processed_files = 0, processed_rows = processed_rows, 
                                                     rows_after_cleaning = 0, execution_time = execution_time, throughput = throughput,
                                                     threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                     benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), 
                                                     resource_stats = resource_stats, chunksize = chunksize, created_parquet_files = 0,
                                                     msg = MSG_ERROR, summary = pd.DataFrame(), missing_values = pd.DataFrame(),
                                                     parquet_files_size_mib = 0.0)

    connection = None
    try: 
        connection = duckdb.connect()
    except duckdb.Error as e:
        msg = f"Failed to initialize DuckDB connection (threads = {threads}): {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        throughput = processed_rows / execution_time if execution_time > 0 else 0.0
        return build_return_dict_parquet_persistence(slice_path = slice_path, parquet_path = parquet_path, operation = CHUNKING, ntest = ntest, processed_files = 0, processed_rows = processed_rows, 
                                                     rows_after_cleaning = 0, execution_time = execution_time, throughput = throughput,
                                                     threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                     benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), 
                                                     resource_stats = resource_stats, chunksize = chunksize, created_parquet_files = 0,
                                                     msg = MSG_ERROR, summary = pd.DataFrame(), missing_values = pd.DataFrame(),
                                                     parquet_files_size_mib = 0.0)
        
    try: 
        processed_files = 0
        created_parquet_files = 0
        parquet_files_size_mib = 0.0
        rows_after_cleaning = 0
        first_chunk_header = True
        missing_counts_total = None
        missing_values = pd.DataFrame()
        
        for f in files:
            print(f"Reading {f}...")
    
            year, month, msg = extract_year_month_from_filename(f)
    
            if year is None or month is None:
                log_error_msg(logger, msg)
                timer.stop("total")
                resource_stats = monitor.stop()
                benchmark_end_timestamp = Timer.get_timestamp()
                execution_time = timer.duration("total")
                throughput = processed_rows / execution_time if execution_time > 0 else 0.0
                return build_return_dict_parquet_persistence(slice_path = slice_path, parquet_path = parquet_path, operation = CHUNKING, ntest = ntest, processed_files = processed_files, processed_rows = processed_rows, 
                                                             rows_after_cleaning = rows_after_cleaning, execution_time = execution_time, throughput = throughput,
                                                             threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                             benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), 
                                                             resource_stats = resource_stats, chunksize = chunksize, created_parquet_files = created_parquet_files,
                                                             msg = MSG_ERROR, summary = pd.DataFrame(), parquet_files_size_mib = parquet_files_size_mib,
                                                             missing_values = finalise_profile_missing_values(missing_counts_total, rows_after_cleaning))
                
            try:
                chunk_iter = pd.read_csv(
                    f,
                    chunksize = chunksize,
                    dtype = {"store_and_fwd_flag": "string"},
                    low_memory = False
                )
            except Exception as e:
                log_error_msg(logger, f"Failed to open file {f} as chunked CSV: {type(e).__name__}: {e}")
                timer.stop("total")
                resource_stats = monitor.stop()
                benchmark_end_timestamp = Timer.get_timestamp()
                execution_time = timer.duration("total")
                throughput = processed_rows / execution_time if execution_time > 0 else 0.0
                return build_return_dict_parquet_persistence(slice_path = slice_path, parquet_path = parquet_path, operation = CHUNKING, ntest = ntest, processed_files = processed_files, processed_rows = processed_rows, 
                                                             rows_after_cleaning = rows_after_cleaning, execution_time = execution_time, throughput = throughput,
                                                             threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                             benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), 
                                                             resource_stats = resource_stats, chunksize = chunksize, 
                                                             created_parquet_files = created_parquet_files, parquet_files_size_mib = parquet_files_size_mib,
                                                             msg = MSG_ERROR, summary = pd.DataFrame(), missing_values = finalise_profile_missing_values(missing_counts_total, rows_after_cleaning))
                            
    
            path = Path(f)
            removed_slice_path = path.parents[2] / REMOVED_FOLDER / path.parent.name
            removed_slice_path.mkdir(parents = True, exist_ok = True)        
            for chunk_id, chunk in enumerate(chunk_iter, start = 1):
                chunk["year"] = year
                chunk["month"] = month
                chunk["currency"] = CURRENCY
                chunk["trip_distance_unit"] = DISTANCE_UNIT
                chunk["suspicious"] = False
                df_normalised = pd.DataFrame()
                if not is_empty(chunk): 
                    # Apply new schema
                    df_normalised = apply_master_schema(chunk, logger)
                df_cleaned = pd.DataFrame()
                if not is_empty(df_normalised):
                    # Clean dataset and transform it
                    df_cleaned = cleaning_transformation(df = df_normalised, removed_slice_path = removed_slice_path, logger = logger, process_type = CHUNKING, header = first_chunk_header)
                    first_chunk_header = False
                rows_after_cleaning += len(df_cleaned)
                # Profiling
                if not is_empty(df_cleaned):
                    missing_counts_total = update_profile_missing_values(df_cleaned, missing_counts_total)
                    persisted, created_parquet_files_aux, parquet_files_size_mib_aux = persist_partitioned_parquet_duckdb(df = df_cleaned, threads = threads, 
                                                                                                                          parquet_path = Path(parquet_path), 
                                                                                                                          logger = logger,
                                                                                                                          verify_count_rows = verify_count_rows,
                                                                                                                          connection = connection)
                    created_parquet_files += created_parquet_files_aux
                    parquet_files_size_mib += parquet_files_size_mib_aux
                    if not persisted: 
                        log_error_msg(logger, "Persistence to parquet failed")
                        timer.stop("total")
                        resource_stats = monitor.stop()
                        benchmark_end_timestamp = Timer.get_timestamp()
                        execution_time = timer.duration("total")
                        throughput = processed_rows / execution_time if execution_time > 0 else 0.0
                        return build_return_dict_parquet_persistence(slice_path = slice_path, parquet_path = parquet_path, operation = CHUNKING, ntest = ntest, processed_files = processed_files, processed_rows = processed_rows, 
                                                                     rows_after_cleaning = rows_after_cleaning, execution_time = execution_time, throughput = throughput,
                                                                     threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                                     benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), 
                                                                     resource_stats = resource_stats, chunksize = chunksize, 
                                                                     created_parquet_files = created_parquet_files, parquet_files_size_mib = parquet_files_size_mib,
                                                                     msg = MSG_ERROR, summary = pd.DataFrame(), missing_values = finalise_profile_missing_values(missing_counts_total, rows_after_cleaning))
                                    
                else:
                    logger.info(f"Chunk {chunk_id} of file {f} produced no cleaned rows to persist")
                processed_rows += len(chunk)
    
                # Cleanup
                del chunk
                del df_normalised
                del df_cleaned
                    
            processed_files += 1
            del chunk_iter
            gc.collect()
        
        missing_values = finalise_profile_missing_values(missing_counts_total, rows_after_cleaning)

    except Exception as e:
        log_error_msg(logger, f"Persistence to parquet completed with errors: {type(e).__name__}: {e}")
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        throughput = processed_rows / execution_time if execution_time > 0 else 0.0
        return build_return_dict_parquet_persistence(slice_path = slice_path, parquet_path = parquet_path, operation = CHUNKING, ntest = ntest, processed_files = processed_files, processed_rows = processed_rows, 
                                                     rows_after_cleaning = rows_after_cleaning, execution_time = execution_time, throughput = throughput,
                                                     threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                     benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), 
                                                     resource_stats = resource_stats, chunksize = chunksize, 
                                                     created_parquet_files = created_parquet_files, parquet_files_size_mib = parquet_files_size_mib,
                                                     msg = MSG_ERROR, summary = pd.DataFrame(), missing_values = finalise_profile_missing_values(missing_counts_total, rows_after_cleaning))
                        
    finally:
        if connection is not None:
            connection.close()

    timer.stop("total")
    resource_stats = monitor.stop()
    benchmark_end_timestamp = Timer.get_timestamp()

    execution_time = timer.duration("total")
    throughput = processed_rows / execution_time if execution_time > 0 else 0
    
    print(f"DuckDB persisted {processed_files} file(s) and {processed_rows} row(s) to Parquet in {execution_time:.2f} seconds")
    print(f"Throughput: {throughput:.2f} rows/sec")

    return build_return_dict_parquet_persistence(slice_path = slice_path, parquet_path = parquet_path, operation = CHUNKING, ntest = ntest, processed_files = processed_files, processed_rows = processed_rows, 
                                                 rows_after_cleaning = rows_after_cleaning, execution_time = execution_time, throughput = throughput,
                                                 threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                 benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), 
                                                 resource_stats = resource_stats, chunksize = chunksize, 
                                                 created_parquet_files = created_parquet_files, parquet_files_size_mib = parquet_files_size_mib,
                                                 msg = "Ingestion, cleaning, transformation, profiling and persistence to parquet completed successfully", summary = pd.DataFrame(), missing_values = missing_values)
                      

In [211]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
result = benchmark_duckdb_parquet_persistence_chunk(slice_path = "../data/CSV/raw-data/2nd-slice/", parquet_path= "../data/parquet/2nd-slice/", 
                                                    logger = logger, ntest = 1,
                                                    verify_count_rows = True, threads = 3)
log_info_msg(logger, result["msg"])

Reading ../data/CSV/raw-data/2nd-slice/yellow_tripdata_2025-05.csv...
Reading ../data/CSV/raw-data/2nd-slice/yellow_tripdata_2025-06.csv...
Reading ../data/CSV/raw-data/2nd-slice/yellow_tripdata_2025-07.csv...
DuckDB persisted 3 file(s) and 12813768 row(s) to Parquet in 573.50 seconds
Throughput: 22343.12 rows/sec


In [212]:
print(result)

{'stage': 'ingestion, cleaning, transformation, profiling and persistence to parquet', 'operation': 'chunking', 'test_number': 1, 'slice_path': '../data/CSV/raw-data/2nd-slice', 'parquet_path': '../data/parquet/2nd-slice/', 'processed_files': 3, 'processed_rows': 12813768, 'rows_after_cleaning': 12621491, 'execution_time_sec': 573.4995542850011, 'throughput_rows_sec': 22343.117626264426, 'threads': 3, 'chunksize': 100000, 'dataset_size_mib': 1321.7953338623047, 'parquet_files_size_mib': 259.01158142089844, 'created_parquet_files': 129, 'benchmark_start_timestamp': '2026-04-10T16:34:14.589098+01:00', 'benchmark_end_timestamp': '2026-04-10T16:43:48.211345+01:00', 'avg_rss_mib': 355.062737358941, 'peak_rss_mib': 417.04296875, 'memory_percent_of_total': 5.31556946092997, 'avg_process_cpu_percent': 133.4208333333334, 'peak_process_cpu_percent': 249.2, 'avg_system_cpu_per_core_percent': [36.38107638888889, 35.25928819444446, 29.90572916666663, 24.497743055555535], 'peak_system_cpu_per_core_p

In [214]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
result = benchmark_duckdb_parquet_persistence_chunk(slice_path = "../data/CSV/raw-data/3rd-slice/", 
                                                    parquet_path= "../data/parquet/3rd-slice/", 
                                                    logger = logger, ntest = 1, 
                                                    verify_count_rows = True, 
                                                    threads = 3)
log_info_msg(logger, result["msg"])

Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-06.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-07.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-08.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-09.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-10.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-11.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2024-12.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-01.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-02.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-03.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-04.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-05.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-06.csv...
Reading ../data/CSV/raw-data/3rd-slice/yellow_tripdata_2025-07.csv...
DuckDB persisted 14 

In [215]:
print(result)

{'stage': 'ingestion, cleaning, transformation, profiling and persistence to parquet', 'operation': 'chunking', 'test_number': 1, 'slice_path': '../data/CSV/raw-data/3rd-slice', 'parquet_path': '../data/parquet/3rd-slice/', 'processed_files': 14, 'processed_rows': 52359167, 'rows_after_cleaning': 52083818, 'execution_time_sec': 2430.9372129959993, 'throughput_rows_sec': 21538.675174366246, 'threads': 3, 'chunksize': 100000, 'dataset_size_mib': 5343.146864891052, 'parquet_files_size_mib': 1066.5396633148193, 'created_parquet_files': 529, 'benchmark_start_timestamp': '2026-04-10T16:48:33.440322+01:00', 'benchmark_end_timestamp': '2026-04-10T17:29:04.480593+01:00', 'avg_rss_mib': 361.1316645174382, 'peak_rss_mib': 446.8515625, 'memory_percent_of_total': 5.69550549266715, 'avg_process_cpu_percent': 134.36658986175084, 'peak_process_cpu_percent': 307.2, 'avg_system_cpu_per_core_percent': [35.40883954754925, 33.14000837871797, 28.441034771679934, 29.195056556346906], 'peak_system_cpu_per_cor

In [216]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
result = benchmark_duckdb_parquet_persistence_chunk(slice_path = "../data/CSV/raw-data/4th-slice/", 
                                                    parquet_path= "../data/parquet/4th-slice/", 
                                                    logger = logger, ntest = 1, 
                                                    verify_count_rows = True, 
                                                    threads = 3)
log_info_msg(logger, result["msg"])

Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-01.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-02.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-03.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-04.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-05.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-06.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-07.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-08.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-09.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-10.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-11.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2023-12.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2024-01.csv...
Reading ../data/CSV/raw-data/4th-slice/yellow_tripdata_2024-02.csv...
Reading ../data/CSV/

In [217]:
print(result)

{'stage': 'ingestion, cleaning, transformation, profiling and persistence to parquet', 'operation': 'chunking', 'test_number': 1, 'slice_path': '../data/CSV/raw-data/4th-slice', 'parquet_path': '../data/parquet/4th-slice/', 'processed_files': 31, 'processed_rows': 107462293, 'rows_after_cleaning': 107143226, 'execution_time_sec': 3407.0957170149995, 'throughput_rows_sec': 31540.73202679175, 'threads': 3, 'chunksize': 100000, 'dataset_size_mib': 10898.749529838562, 'parquet_files_size_mib': 2193.3621978759766, 'created_parquet_files': 1091, 'benchmark_start_timestamp': '2026-04-10T17:40:58.306955+01:00', 'benchmark_end_timestamp': '2026-04-10T18:37:45.515169+01:00', 'avg_rss_mib': 365.3100536505203, 'peak_rss_mib': 472.125, 'memory_percent_of_total': 6.017637077693956, 'avg_process_cpu_percent': 126.9120359955003, 'peak_process_cpu_percent': 265.8, 'avg_system_cpu_per_core_percent': [36.75248875140601, 32.918278965129325, 32.6207395950507, 24.385376827896664], 'peak_system_cpu_per_core_

##### Number of rows and mbytes

In [63]:
files = glob.glob("../data/parquet/2nd-slice/year=2025/**/*.parquet", recursive = True)

print(len(files))

df_check = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

print(df_check.shape)

129
(12621541, 23)


In [64]:
# WHERE suspicious = False
rows = duckdb.sql("SELECT COUNT(*) FROM read_parquet('../data/parquet/4th-slice/')").fetchone()[0]
print(rows)

107144513


In [55]:
from pathlib import Path

def folder_size_mb(path):
    total = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())
    return total / (1024 * 1024)

print(folder_size_mb("../data/parquet/4th-slice/"))

2193.3621978759766


#### Compaction stage

In [51]:
from datetime import datetime
from uuid import uuid4

In [52]:
# Count parquet files in a given directory recursively
# Parameters:
# parquet_path: directory path containing parquet files (default behaviour) OR glob pattern if use_pattern = True
# use_pattern: if parquet_path is a glob pattern
# Returns:
# number of parquet files found
def count_parquet_files(
    parquet_path: str | Path,
    use_pattern: bool = False
) -> int:
    
    if use_pattern:
        return len(list(Path().glob(str(parquet_path))))             
    else: 
        path = Path(parquet_path)
        if not path.exists():
            return 0
            
        return sum(1 for _ in path.rglob("*.parquet"))

In [53]:
# Build benchmark return for compaction to one parquet file per partition step
def build_return_dict_compaction_parquet_lake(
    parquet_path: str | Path,
    compacted_parquet_path: str | Path,
    operation: str,
    ntest: int,
    created_parquet_files: int,
    total_partitions: int,
    compacted_parquet_size: float,
    processed_files: int,
    processed_rows: int,
    execution_time: float,
    throughput: float,
    threads: int,
    benchmark_start_timestamp: str,
    benchmark_end_timestamp: str,
    msg: str,
    resource_stats: dict[str, Any] | None = None,
) -> dict[str, Any]:
    
    if resource_stats is None:
        resource_stats = ResourceMonitor.as_dict()

    return {
        # operations:
        "stage": COMPACTION_TO_PARQUET_FILE,
        # Type of operation
        "operation": operation,
        # Test Number
        "test_number": ntest,  
        # Parquet paths
        "parquet_path": str(parquet_path),
        "compacted_parquet_path": str(compacted_parquet_path),
        # Performance
        "processed_files": processed_files,
        "processed_rows": processed_rows,
        "execution_time_sec": execution_time,
        "throughput_rows_sec": throughput,
        "threads": threads,
        "parquet_size_mib": compute_dataset_size_mib(parquet_path),
        "compacted_parquet_size_mib": compacted_parquet_size,
        "compacted_parquet_files_created": created_parquet_files,
        "total_partitions_created": total_partitions,
        # Timestamps
        "benchmark_start_timestamp": benchmark_start_timestamp,
        "benchmark_end_timestamp": benchmark_end_timestamp,
        # Resources
        "avg_rss_mib": resource_stats["avg_rss_mib"],
        "peak_rss_mib": resource_stats["peak_rss_mib"],
        "memory_percent_of_total": resource_stats["memory_percent_of_total"],
        "avg_process_cpu_percent": resource_stats["avg_process_cpu_percent"],
        "peak_process_cpu_percent": resource_stats["peak_process_cpu_percent"],
        "avg_system_cpu_per_core_percent": resource_stats["avg_system_cpu_per_core_percent"],
        "peak_system_cpu_per_core_percent": resource_stats["peak_system_cpu_per_core_percent"],
        "read_mib": resource_stats["read_mib"],
        "write_mib": resource_stats["write_mib"],    
        # Message
        "msg": msg
    }

In [54]:
# Benchamrk compaction of partitioned parquet lake using duckdb
# Parameters:
# parquet_path: directory partitioned parquet lake
# compacted_parquet_path: output path for compacted parquet lake
# ntest: test number
# threads: number of DuckDB execution threads
# logger: Logger instance for reporting progress and errors
# verify_count_rows: if to verify row counts after compaction
# Returns:
# benchmark results including parquet path, compacted parquet path, processed files, processed rows, execution time, 
# throughput, threads, timestamps, resource statistics and a message
def benchmark_compaction_parquet_lake(
    parquet_path: str | Path, 
    compacted_parquet_path: str | Path, 
    ntest: int,
    threads: int = 0, 
    logger: logging.Logger | None = None,
    verify_count_rows: bool = False
) -> dict[str, Any]:

    if logger is None:
        logger = setup_logger()
                
    timer = Timer()
    monitor = ResourceMonitor()
    benchmark_start_timestamp = Timer.get_timestamp()
    timer.start("total")
    monitor.start()
    
    parquet_path = Path(parquet_path)
    compacted_parquet_path = Path(compacted_parquet_path)
    created_parquet_files = 0
    total_rows = 0
    total_files = 0
    total_partitions = 0
    compacted_parquet_size = 0
    MSG_ERROR = "Compaction step completed with errors, check pipeline.log"
    MIB = 1024 * 1024

    if not parquet_path.exists():
        msg = f"Source parquet path does not exist: {parquet_path}"
        log_error_msg(logger, msg)
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        throughput = total_rows / execution_time if execution_time > 0 else 0.0
        return build_return_dict_compaction_parquet_lake(parquet_path = parquet_path, compacted_parquet_path = compacted_parquet_path,
                                                         operation = STANDARD, ntest = ntest, processed_files = total_files, processed_rows = total_rows,
                                                         execution_time = execution_time, throughput = throughput, threads = threads,
                                                         benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                         benchmark_end_timestamp =  benchmark_end_timestamp.isoformat(),
                                                         resource_stats = resource_stats, created_parquet_files = created_parquet_files,
                                                         msg = MSG_ERROR, total_partitions = total_partitions,
                                                         compacted_parquet_size = compacted_parquet_size)

    try:
        connection = duckdb.connect()
        if threads and threads > 0:
            connection.execute(f"PRAGMA threads={threads}")
    except duckdb.Error as e:
        msg = f"Failed to initialize DuckDB connection (threads = {threads}): {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        if connection is not None:
            connection.close()
            
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        throughput = total_rows / execution_time if execution_time > 0 else 0.0
        return build_return_dict_compaction_parquet_lake(parquet_path = parquet_path, compacted_parquet_path = compacted_parquet_path,
                                                         operation = STANDARD, ntest = ntest, processed_files = total_files, processed_rows = total_rows,
                                                         execution_time = execution_time, throughput = throughput, threads = threads,
                                                         benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                         benchmark_end_timestamp =  benchmark_end_timestamp.isoformat(),
                                                         resource_stats = resource_stats, created_parquet_files = created_parquet_files,
                                                         msg = MSG_ERROR, total_partitions = total_partitions,
                                                         compacted_parquet_size = compacted_parquet_size)
            
    try: 

        for year_dir in parquet_path.glob("year=*"):
            for month_dir in year_dir.glob("month=*"):
    
                rel_path = month_dir.relative_to(parquet_path)
                target_dir = compacted_parquet_path / rel_path
                
                existing_files = set(target_dir.rglob("*.parquet"))
                
                target_dir.mkdir(parents = True, exist_ok = True)

                run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

                target_file = target_dir / f"compacted_data_{run_id}_{uuid4().hex}.parquet"
    
                pattern_read = (month_dir / "*.parquet").as_posix()

                total_files += count_parquet_files(parquet_path = pattern_read, use_pattern = True)
        
                nrows = connection.execute(f"SELECT COUNT(*) FROM read_parquet('{pattern_read}')").fetchone()[0]
                
                query = f"""
                COPY (
                    SELECT *
                    FROM read_parquet('{pattern_read}')
                ) 
                TO '{target_file.as_posix()}'
                (FORMAT PARQUET);
                """
            
                connection.execute(query)
                total_partitions += 1
                new_files = set(target_dir.rglob("*.parquet"))
                # This logic was add, because it can be extended to the creation of more than one file
                created_files = new_files - existing_files
                
                if not created_files:
                    msg = f"No new parquet file was created in: {target_dir}"
                    log_error_msg(logger, msg)
                    timer.stop("total")
                    resource_stats = monitor.stop()
                    benchmark_end_timestamp = Timer.get_timestamp()
                    execution_time = timer.duration("total")
                    throughput = total_rows / execution_time if execution_time > 0 else 0.0
                    return build_return_dict_compaction_parquet_lake(parquet_path = parquet_path, compacted_parquet_path = compacted_parquet_path,
                                                                     operation = STANDARD, ntest = ntest, processed_files = total_files, processed_rows = total_rows,
                                                                     execution_time = execution_time, throughput = throughput, threads = threads,
                                                                     benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                                     benchmark_end_timestamp =  benchmark_end_timestamp.isoformat(),
                                                                     resource_stats = resource_stats, created_parquet_files = created_parquet_files,
                                                                     msg = MSG_ERROR, total_partitions = total_partitions,
                                                                     compacted_parquet_size = compacted_parquet_size)

                created_parquet_files += len(created_files)
                created_paths = [f.as_posix() for f in created_files]
                paths_sql = ", ".join(f"'{p}'" for p in created_paths)

                if verify_count_rows:
                    readback_nrows = connection.execute(f"SELECT COUNT(*) FROM read_parquet([{paths_sql}])").fetchone()[0]
                    if nrows != readback_nrows:
                        msg = f"Row mismatch in {rel_path}: Expected {nrows} rows, but new parquet files contain {readback_nrows}"
                        log_error_msg(logger, msg)
                        timer.stop("total")
                        resource_stats = monitor.stop()
                        benchmark_end_timestamp = Timer.get_timestamp()
                        execution_time = timer.duration("total")
                        throughput = total_rows / execution_time if execution_time > 0 else 0.0
                        return build_return_dict_compaction_parquet_lake(parquet_path = parquet_path, compacted_parquet_path = compacted_parquet_path,
                                                                         operation = STANDARD, ntest = ntest, processed_files = total_files, processed_rows = total_rows,
                                                                         execution_time = execution_time, throughput = throughput, threads = threads,
                                                                         benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                                         benchmark_end_timestamp =  benchmark_end_timestamp.isoformat(),
                                                                         resource_stats = resource_stats, created_parquet_files = created_parquet_files,
                                                                         msg = MSG_ERROR, total_partitions = total_partitions,
                                                                         compacted_parquet_size = compacted_parquet_size)                

                size_bytes = [f.stat().st_size for f in created_files]
                compacted_parquet_size += (sum(size_bytes) / MIB)
                
                total_rows += nrows
                msg = f"Compacted {rel_path}. ({nrows} row(s))"
                log_info_msg(logger, msg)
                
    except Exception as e:
        msg = f"Compaction failed: {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        throughput = total_rows / execution_time if execution_time > 0 else 0.0
        return build_return_dict_compaction_parquet_lake(parquet_path = parquet_path, compacted_parquet_path = compacted_parquet_path,
                                                         operation = STANDARD, ntest = ntest, processed_files = total_files, processed_rows = total_rows,
                                                         execution_time = execution_time, throughput = throughput, threads = threads,
                                                         benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                         benchmark_end_timestamp =  benchmark_end_timestamp.isoformat(),
                                                         resource_stats = resource_stats, created_parquet_files = created_parquet_files,
                                                         msg = MSG_ERROR, total_partitions = total_partitions,
                                                         compacted_parquet_size = compacted_parquet_size)

                
    finally:
        connection.close()
                               
    timer.stop("total")
    resource_stats = monitor.stop()
    benchmark_end_timestamp = Timer.get_timestamp()
    execution_time = timer.duration("total")
    throughput = total_rows / execution_time if execution_time > 0 else 0
            
    print(f"DuckDB compacted {total_rows} row(s) to a Parquet file in {execution_time:.2f} seconds")
    print(f"Throughput: {throughput:.2f} rows/sec")

    return build_return_dict_compaction_parquet_lake(parquet_path = parquet_path, compacted_parquet_path = compacted_parquet_path,
                                                     operation = STANDARD, ntest = ntest, processed_files = total_files, processed_rows = total_rows,
                                                     execution_time = execution_time, throughput = throughput, threads = threads,
                                                     benchmark_start_timestamp = benchmark_start_timestamp.isoformat(),
                                                     benchmark_end_timestamp =  benchmark_end_timestamp.isoformat(),
                                                     resource_stats = resource_stats, created_parquet_files = created_parquet_files,
                                                     msg = "Compaction step completed successfully", total_partitions = total_partitions,
                                                     compacted_parquet_size = compacted_parquet_size)
    

In [55]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()

result = benchmark_compaction_parquet_lake(parquet_path = Path("../data/parquet/1st-slice/"), 
                                           compacted_parquet_path = "../data/parquet/1st-slice_compacted/", 
                                           logger = logger, ntest = 1,
                                           verify_count_rows = True, threads = 3)
log_info_msg(logger, result["msg"])

DuckDB compacted 3842004 row(s) to a Parquet file in 7.42 seconds
Throughput: 517674.03 rows/sec


In [56]:
print(result)

{'stage': 'compaction to one parquet file per partition', 'operation': 'standard', 'test_number': 1, 'parquet_path': '../data/parquet/1st-slice', 'compacted_parquet_path': '../data/parquet/1st-slice_compacted', 'processed_files': 1, 'processed_rows': 3842004, 'execution_time_sec': 7.421666522999999, 'throughput_rows_sec': 517674.0275372786, 'threads': 3, 'parquet_size_mib': 79.21061992645264, 'compacted_parquet_size_mib': 79.2763671875, 'compacted_parquet_files_created': 1, 'total_partitions_created': 1, 'benchmark_start_timestamp': '2026-04-11T16:35:11.702947+01:00', 'benchmark_end_timestamp': '2026-04-11T16:35:19.128523+01:00', 'avg_rss_mib': 428.4681755514706, 'peak_rss_mib': 515.80078125, 'memory_percent_of_total': 6.574309185959671, 'avg_process_cpu_percent': 300.8705882352941, 'peak_process_cpu_percent': 364.6, 'avg_system_cpu_per_core_percent': [53.41470588235295, 86.96176470588235, 83.33235294117648, 70.58235294117648], 'peak_system_cpu_per_core_percent': [100.0, 100.0, 100.0, 

In [58]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()

result = benchmark_compaction_parquet_lake(parquet_path = Path("../data/parquet/2nd-slice/"), 
                                           compacted_parquet_path = "../data/parquet/2nd-slice_compacted/", 
                                           logger = logger, ntest = 1, 
                                           verify_count_rows = True,
                                           threads = 3)
log_info_msg(logger, result["msg"])

DuckDB compacted 12621491 row(s) to a Parquet file in 29.10 seconds
Throughput: 433752.27 rows/sec


In [59]:
print(result)

{'stage': 'compaction to one parquet file per partition', 'operation': 'standard', 'test_number': 1, 'parquet_path': '../data/parquet/2nd-slice', 'compacted_parquet_path': '../data/parquet/2nd-slice_compacted', 'processed_files': 129, 'processed_rows': 12621491, 'execution_time_sec': 29.098386043000062, 'throughput_rows_sec': 433752.2700176094, 'threads': 3, 'parquet_size_mib': 259.01158142089844, 'compacted_parquet_size_mib': 260.64246940612793, 'compacted_parquet_files_created': 3, 'total_partitions_created': 3, 'benchmark_start_timestamp': '2026-04-11T16:39:12.082272+01:00', 'benchmark_end_timestamp': '2026-04-11T16:39:41.320711+01:00', 'avg_rss_mib': 479.3411896008403, 'peak_rss_mib': 552.046875, 'memory_percent_of_total': 7.03629574309186, 'avg_process_cpu_percent': 286.933613445378, 'peak_process_cpu_percent': 364.8, 'avg_system_cpu_per_core_percent': [77.27647058823531, 58.74453781512605, 59.94285714285712, 83.01092436974787], 'peak_system_cpu_per_core_percent': [100.0, 100.0, 1

In [60]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()

result = benchmark_compaction_parquet_lake(parquet_path = Path("../data/parquet/3rd-slice/"), 
                                           compacted_parquet_path = "../data/parquet/3rd-slice_compacted/", 
                                           logger = logger, ntest = 1, 
                                           verify_count_rows = True,
                                           threads = 3)
log_info_msg(logger, result["msg"])

DuckDB compacted 52083818 row(s) to a Parquet file in 133.18 seconds
Throughput: 391064.08 rows/sec


In [61]:
print(result)

{'stage': 'compaction to one parquet file per partition', 'operation': 'standard', 'test_number': 1, 'parquet_path': '../data/parquet/3rd-slice', 'compacted_parquet_path': '../data/parquet/3rd-slice_compacted', 'processed_files': 529, 'processed_rows': 52083818, 'execution_time_sec': 133.18486908399996, 'throughput_rows_sec': 391064.07776059484, 'threads': 3, 'parquet_size_mib': 1066.5396633148193, 'compacted_parquet_size_mib': 1072.089677810669, 'compacted_parquet_files_created': 14, 'total_partitions_created': 14, 'benchmark_start_timestamp': '2026-04-11T16:43:42.061459+01:00', 'benchmark_end_timestamp': '2026-04-11T16:45:55.253290+01:00', 'avg_rss_mib': 552.206532248004, 'peak_rss_mib': 836.89453125, 'memory_percent_of_total': 10.666915608663182, 'avg_process_cpu_percent': 284.0249500998008, 'peak_process_cpu_percent': 379.0, 'avg_system_cpu_per_core_percent': [73.99241516966067, 67.02095808383234, 70.97884231536928, 67.7906187624751], 'peak_system_cpu_per_core_percent': [100.0, 100

In [63]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()

result = benchmark_compaction_parquet_lake(parquet_path = Path("../data/parquet/4th-slice/"), 
                                           compacted_parquet_path = "../data/parquet/4th-slice_compacted/", 
                                           logger = logger, ntest = 1, 
                                           verify_count_rows = True,
                                           threads = 3)
log_info_msg(logger, result["msg"])

DuckDB compacted 107143226 row(s) to a Parquet file in 249.39 seconds
Throughput: 429626.42 rows/sec


In [64]:
print(result)

{'stage': 'compaction to one parquet file per partition', 'operation': 'standard', 'test_number': 1, 'parquet_path': '../data/parquet/4th-slice', 'compacted_parquet_path': '../data/parquet/4th-slice_compacted', 'processed_files': 1091, 'processed_rows': 107143226, 'execution_time_sec': 249.3869565529999, 'throughput_rows_sec': 429626.4226522603, 'threads': 3, 'parquet_size_mib': 2193.3621978759766, 'compacted_parquet_size_mib': 2198.7793397903442, 'compacted_parquet_files_created': 31, 'total_partitions_created': 31, 'benchmark_start_timestamp': '2026-04-11T16:51:40.818437+01:00', 'benchmark_end_timestamp': '2026-04-11T16:55:50.213826+01:00', 'avg_rss_mib': 608.7909157372144, 'peak_rss_mib': 853.4765625, 'memory_percent_of_total': 10.878267363704257, 'avg_process_cpu_percent': 293.04657236126206, 'peak_process_cpu_percent': 385.5, 'avg_system_cpu_per_core_percent': [73.57105549510338, 74.1284004352557, 73.46452665941246, 68.32937976060936], 'peak_system_cpu_per_core_percent': [100.0, 1

#### Analytical operations stage (queries)

In [110]:
PARQUET_PATHS = {
    "1st-slice": "../data/parquet/1st-slice_compacted/**/*.parquet",
    "2nd-slice": "../data/parquet/2nd-slice_compacted/**/*.parquet",
    "3rd-slice": "../data/parquet/3rd-slice_compacted/**/*.parquet",
    "4th-slice": "../data/parquet/4th-slice_compacted/**/*.parquet",
}

PARQUET_FOLDERS = {
    "1st-slice": "../data/parquet/1st-slice_compacted/",
    "2nd-slice": "../data/parquet/2nd-slice_compacted/",
    "3rd-slice": "../data/parquet/3rd-slice_compacted/",
    "4th-slice": "../data/parquet/4th-slice_compacted/",
}

DUCKDB_QUERIES = {
    
    "q1_monthly_payment_statistics": """
            SELECT year, month, payment_type_id, COUNT(*) AS trip_count, AVG(trip_distance) AS avg_trip_distance,
                   trip_distance_unit AS trip_distance_unit, AVG(fare_amount) AS avg_fare_amount, AVG(tip_amount) AS avg_tip_amount, 
                   SUM(total_amount) AS sum_total_amount, currency AS currency
             FROM read_parquet('{parquet_path}')
            WHERE suspicious = FALSE
         GROUP BY year, month, payment_type_id, trip_distance_unit, currency
         ORDER BY year, month, payment_type_id
    """,

    "q2_most_valuable_routes": """
            SELECT pickup_location_id, dropoff_location_id, COUNT(*) AS trip_count, AVG(trip_distance) AS avg_trip_distance,
                   trip_distance_unit AS trip_distance_unit, AVG(total_amount) AS avg_total_amount, SUM(total_amount) AS sum_total_amount, 
                   currency AS currency
              FROM read_parquet('{parquet_path}')
          GROUP BY pickup_location_id, dropoff_location_id, trip_distance_unit, currency
            HAVING COUNT(*) > 100
          ORDER BY sum_total_amount DESC, trip_count DESC
             LIMIT 100
    """,

    "q3_hourly_demand_and_speed": """
            WITH BASE AS (
                SELECT EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour, trip_distance, trip_distance_unit, fare_amount, total_amount,
                       currency,
                       EXTRACT(EPOCH FROM (dropoff_datetime - pickup_datetime)) / 60.0 AS trip_duration_min
                  FROM read_parquet('{parquet_path}')
                 WHERE trip_distance > 0
            ) 
            SELECT pickup_hour, COUNT(*) AS trip_count, AVG(trip_distance) AS avg_trip_distance, trip_distance_unit AS trip_distance_unit,
                   AVG(fare_amount) AS avg_fare_amount, AVG(total_amount) AS avg_total_amount, currency AS currency,
                   AVG(trip_distance / NULLIF(trip_duration_min / 60.0, 0)) AS avg_speed_mph
              FROM base
          GROUP BY pickup_hour, trip_distance_unit, currency
          ORDER BY pickup_hour, trip_distance_unit, currency
    """
}

FULL_DATASET_SIZE_QUERY = "SELECT COUNT(*) FROM read_parquet('{parquet_path}')"


In [134]:
# Get the total number of rows in a parquet lake using DuckDB
def get_total_size_dataset(
    parquet_path: str | Path, 
    threads: int = 0, 
    logger: logging.Logger | None = None,
) -> tuple[int, bool]:

    if logger is None:
        logger = setup_logger()
    
    if "{parquet_path}" not in FULL_DATASET_SIZE_QUERY:
        msg = f"Query to get full dataset size does not contain '{{parquet_path}}': '{parquet_path}'"
        log_error_msg(logger, msg)
        return 0, False

    try:
        query_full_dataset_size = FULL_DATASET_SIZE_QUERY.format(parquet_path = str(parquet_path))
    except Exception as e:
        msg = f"Error formatting query to get full dataset size: {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        return 0, False
        
    connection = None
    try:
        connection = duckdb.connect()
        if threads and threads > 0:
            connection.execute(f"PRAGMA threads={threads}")
    except Exception as e:
        msg = f"Failed to initialize DuckDB connection (threads = {threads}): {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        if connection is not None:
            connection.close()
        return 0, False
                            
    try:
        total_rows = connection.execute(query_full_dataset_size).fetchone()[0]
    except Exception as e:
        msg = f"Query to get full dataset size could not be executed due to the following error: {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        return 0, False
    finally:
        connection.close()

    return total_rows, True

In [116]:
# Build benchmark return for queries step
def build_return_dict_queries(
    slice_name: str,
    parquet_folder: str | Path,
    operation: str,
    ntest: int,
    query_name: str,
    total_files: int,
    total_rows_in_dataset: int,
    rows_returned: int,
    execution_time: float,
    throughput: float,
    threads: int,
    benchmark_start_timestamp: str,
    benchmark_end_timestamp: str,
    msg: str,
    resource_stats: dict[str, Any] | None = None,
) -> dict[str, Any]:

    if resource_stats is None:
        resource_stats = ResourceMonitor.as_dict()

    return {
            # operations:
            "stage": QUERIES,
            # Type of operation
            "operation": operation,
            # Test Number
            "test_number": ntest,  
            #Slice name
            "slice_name": slice_name,
            # Slice path
            "parquet_path": str(parquet_folder),
            # Query name
            "query_name": query_name,
            # Performance
            "total_files": total_files,
            "total_rows_in_dataset": total_rows_in_dataset,
            "rows_returned": rows_returned,
            "execution_time_sec": execution_time,
            "throughput_rows_sec": throughput,
            "threads": threads,
            "parquet_files_size_mib": compute_dataset_size_mib(parquet_folder),
            # Timestamps
            "benchmark_start_timestamp": benchmark_start_timestamp,
            "benchmark_end_timestamp": benchmark_end_timestamp,
            # Resources
            "avg_rss_mib": resource_stats["avg_rss_mib"],
            "peak_rss_mib": resource_stats["peak_rss_mib"],
            "memory_percent_of_total": resource_stats["memory_percent_of_total"],
            "avg_process_cpu_percent": resource_stats["avg_process_cpu_percent"],
            "peak_process_cpu_percent": resource_stats["peak_process_cpu_percent"],
            "avg_system_cpu_per_core_percent": resource_stats["avg_system_cpu_per_core_percent"],
            "peak_system_cpu_per_core_percent": resource_stats["peak_system_cpu_per_core_percent"],
            "read_mib": resource_stats["read_mib"],
            "write_mib": resource_stats["write_mib"], 
            # Message
            "msg":  msg
    }

In [135]:
# Benchmark DuckDB queries against a parquet dataset
# Parameters:
# slice_name: key used to resolve the parquet dataset path from PARQUET_PATHS
# query_name: key used to resolve the query template from DUCKDB_QUERIES
# ntest: test number
# threads: number of DuckDB execution threads
# logger: Logger instance for reporting progress and errors
# Returns:
# benchmark results including slice name, parquet path, query name, total files, total rows in dataset, execution time, 
# throughput, threads, timestamps, resource statistics and a message
def benchmark_duckdb_queries(
    slice_name: str, 
    query_name: str, 
    ntest: int,
    threads: int = 0, 
    logger: logging.Logger | None = None,
) -> dict[str, Any]:

    if logger is None:
        logger = setup_logger()
        
    timer = Timer()
    benchmark_start_timestamp = Timer.get_timestamp()
    MSG_ERROR = "Query processed with errors, check pipeline.log"
    parquet_path = PARQUET_PATHS[slice_name]
    query_template = DUCKDB_QUERIES[query_name]
    parquet_folder = PARQUET_FOLDERS[slice_name]

    total_files = count_parquet_files(parquet_path = parquet_path, use_pattern = True)
    
    if total_files == 0:
        msg = f"No parquet file(s) were found for that path: {parquet_folder} OR {parquet_folder} does not exist"
        log_error_msg(logger, msg)
        benchmark_end_timestamp = Timer.get_timestamp()
        return build_return_dict_queries(slice_name = slice_name, parquet_path = parquet_folder, query_name = query_name, 
                                         operation = STANDARD, ntest = ntest, total_files = total_files, total_rows_in_dataset = 0,
                                         rows_returned = 0, execution_time = 0.0, throughput = 0.0, threads = threads, 
                                         benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                         benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                         msg = MSG_ERROR)
    
    if "{parquet_path}" not in query_template:
        msg = f"Query does not contain '{{parquet_path}}': '{parquet_path}'"
        log_error_msg(logger, msg)
        benchmark_end_timestamp = Timer.get_timestamp()
        return build_return_dict_queries(slice_name = slice_name, parquet_path = parquet_folder, query_name = query_name, 
                                         operation = STANDARD, ntest = ntest, total_files = total_files, total_rows_in_dataset = 0,
                                         rows_returned = 0, execution_time = 0.0, throughput = 0.0, threads = threads, 
                                         benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                         benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                         msg = MSG_ERROR)
                
    try:
        query = query_template.format(parquet_path = parquet_path)
        print(f"Query: \n {query}")
    except Exception as e:
        msg = f"Error formatting query {query_name}: {e}"
        log_error_msg(logger, msg)
        benchmark_end_timestamp = Timer.get_timestamp()
        return build_return_dict_queries(slice_name = slice_name, parquet_path = parquet_folder, query_name = query_name, 
                                         operation = STANDARD, ntest = ntest, total_files = total_files, total_rows_in_dataset = 0,
                                         rows_returned = 0, execution_time = 0.0, throughput = 0.0, threads = threads, 
                                         benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                         benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                         msg = MSG_ERROR)

    total_rows = 0
    # Start benchmark
    monitor = ResourceMonitor()
    timer.start("total")
    monitor.start()

    connection = None
    
    try:
        connection = duckdb.connect()
        if threads and threads > 0:
            connection.execute(f"PRAGMA threads={threads}")
    except Exception as e:
        msg = f"Failed to initialize DuckDB connection (threads = {threads}): {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        if connection is not None:
            connection.close()
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        throughput = total_rows / execution_time if execution_time > 0 else 0.0
        return build_return_dict_queries(slice_name = slice_name, parquet_path = parquet_folder, query_name = query_name, 
                                         operation = STANDARD, ntest = ntest, total_files = total_files, total_rows_in_dataset = total_rows,
                                         rows_returned = 0, execution_time = execution_time, throughput = throughput, threads = threads, 
                                         benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                         benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                         msg = MSG_ERROR, resource_stats = resource_stats)
    try:
        rows_returned = connection.execute(f"SELECT COUNT(*) FROM ({query}) t").fetchone()[0]
    except Exception as e:
        msg = f"Query could not be executed due to the following error: {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        throughput = total_rows / execution_time if execution_time > 0 else 0.0
        return build_return_dict_queries(slice_name = slice_name, parquet_path = parquet_folder, query_name = query_name, 
                                         operation = STANDARD, ntest = ntest, total_files = total_files, total_rows_in_dataset = total_rows,
                                         rows_returned = 0, execution_time = execution_time, throughput = throughput, threads = threads, 
                                         benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                         benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                         msg = MSG_ERROR, resource_stats = resource_stats)                                               
    finally:
        if connection is not None:
            connection.close()

        
    timer.stop("total")
    resource_stats = monitor.stop()
    benchmark_end_timestamp = Timer.get_timestamp()
    execution_time = timer.duration("total")
    
    total_rows, query_succeded = get_total_size_dataset(parquet_path = parquet_path, logger = logger)

    if not query_succeded:
        return build_return_dict_queries(slice_name = slice_name, parquet_path = parquet_folder, query_name = query_name, 
                                         operation = STANDARD, ntest = ntest, total_files = total_files, total_rows_in_dataset = total_rows,
                                         rows_returned = rows_returned, execution_time = execution_time, throughput = 0, threads = threads, 
                                         benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                         benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                         msg = MSG_ERROR)
    
    throughput = total_rows / execution_time if execution_time > 0 else 0.0
    print(f"DuckDB executed the query {query_name} in {execution_time:.2f} seconds. Processed {total_rows} row(s) and returned {rows_returned} row(s)")
    print(f"Throughput: {throughput:.2f} rows/sec")

    return build_return_dict_queries(slice_name = slice_name, parquet_path = parquet_folder, query_name = query_name, 
                                     operation = STANDARD, ntest = ntest, total_files = total_files, total_rows_in_dataset = total_rows,
                                     rows_returned = rows_returned, execution_time = execution_time, throughput = throughput, threads = threads, 
                                     benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                     benchmark_end_timestamp = benchmark_end_timestamp.isoformat(),
                                     msg = "Query was processed successfully", resource_stats = resource_stats)
    

In [120]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
results = benchmark_duckdb_queries(slice_name = "1st-slice", query_name = "q3_hourly_demand_and_speed", 
                                   logger = logger, ntest = 1, threads = 3)
log_info_msg(logger, result["msg"])

NameError: name 'count_parquet_files' is not defined

In [149]:
print(results)

{'stage': 'queries', 'operation': 'standard', 'test_number': 1, 'slice_name': '1st-slice', 'parquet_path': '../data/parquet/1st-slice_compacted/**/*.parquet', 'query_name': 'q3_hourly_demand_and_speed', 'total_files': 1, 'total_rows_in_dataset': 3842004, 'rows_returned': 24, 'execution_time_sec': 0.285202641000069, 'throughput_rows_sec': 13471137.52708577, 'threads': 3, 'parquet_files_size_mib': 79.10484504699707, 'benchmark_start_timestamp': '2026-04-09T01:31:54.000536+01:00', 'benchmark_end_timestamp': '2026-04-09T01:31:54.380716+01:00', 'avg_rss_mib': 217.650390625, 'peak_rss_mib': 226.60546875, 'memory_percent_of_total': 2.888279146048015, 'avg_process_cpu_percent': 201.6, 'peak_process_cpu_percent': 249.2, 'avg_system_cpu_per_core_percent': [87.5, 65.4, 87.5, 87.5], 'peak_system_cpu_per_core_percent': [100.0, 100.0, 100.0, 100.0], 'read_mib': 0.734375, 'write_mib': 0.0, 'msg': 'Query was processed successfuly'}


In [140]:
# Table metadata
con = duckdb.connect()

df_meta = con.execute("""
    SELECT * 
    FROM parquet_metadata('../data/parquet/4th-slice_compacted/year=2023/month=4/compacted_data_20260331_190714_028afdc2648e482cbdae86755f73a285.parquet')
""").df()

print(df_meta)

con.close()

                                             file_name  row_group_id  \
0    ../data/parquet/4th-slice_compacted/year=2023/...             0   
1    ../data/parquet/4th-slice_compacted/year=2023/...             0   
2    ../data/parquet/4th-slice_compacted/year=2023/...             0   
3    ../data/parquet/4th-slice_compacted/year=2023/...             0   
4    ../data/parquet/4th-slice_compacted/year=2023/...             0   
..                                                 ...           ...   
670  ../data/parquet/4th-slice_compacted/year=2023/...            26   
671  ../data/parquet/4th-slice_compacted/year=2023/...            26   
672  ../data/parquet/4th-slice_compacted/year=2023/...            26   
673  ../data/parquet/4th-slice_compacted/year=2023/...            26   
674  ../data/parquet/4th-slice_compacted/year=2023/...            26   

     row_group_num_rows  row_group_num_columns  row_group_bytes  column_id  \
0                124492                     25          3

In [141]:
# How to change the row group size
#OLD:
#124K rows → 3.5 MB

#NEW:
#500K rows → ~14 MB

query = f"""
COPY (
    SELECT *
    FROM read_parquet('{""}')
)
TO '{Path("").as_posix()}'
(
    FORMAT PARQUET,
    ROW_GROUP_SIZE 500000
);
"""

#### Machine Learning stage

In [81]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np

##### Training, all data in memory

In [82]:
# Transform features and creates new features based on pickup_datetime and dropoff_datetime
def transform_ml_features(
    df: pd.DataFrame
) -> pd.DataFrame:

    df = df.copy()

    # Pickup datetime
    df["pickup_hour"] = df["pickup_datetime"].dt.hour.astype("UInt8")
    df["pickup_day"] = df["pickup_datetime"].dt.day.astype("UInt8")
    df["pickup_weekday"] = df["pickup_datetime"].dt.weekday.astype("UInt8")
            
    # Dropoff datetime
    df["dropoff_hour"] = df["dropoff_datetime"].dt.hour.astype("UInt8")
    df["dropoff_day"] = df["dropoff_datetime"].dt.day.astype("UInt8")
    df["dropoff_weekday"] = df["dropoff_datetime"].dt.weekday.astype("UInt8")
            
    # Trip duration sec
    df["trip_duration_sec"] = (df["dropoff_datetime"] - df["pickup_datetime"]).dt.total_seconds().astype("UInt32")

    # Trip speed
    df["trip_speed"] = df["trip_distance"] / (df["trip_duration_sec"] / 3600)

    # Drop no numerical columns
    df = df.drop(columns = ["pickup_datetime", "dropoff_datetime"])

    return df
    

In [126]:
# Remove negative monetary values and outliers for ml
def remove_negative_monetary_value_and_outliers_for_ml(
    df: pd.DataFrame
) -> pd.DataFrame:

    mask = (
        df["fare_amount"].ge(0) &
        df["tip_amount"].ge(0) &
        df["fare_amount"].lt(500) & # remove outliers
        df["tip_amount"].lt(200)
    )

    return df[mask].copy()

In [122]:
# Build benchmark return for machine learning step
def build_return_dict_ml(
    slice_name: str,
    operation: str,
    parquet_folder: str | Path,
    ntest: int,
    total_files: int,
    total_rows_in_dataset: int,
    total_rows_used_ml: int,
    execution_time: float,
    execution_time_train: float,
    execution_time_pred: float,
    throughput: float,
    throughput_ml: float, 
    threads: int,
    benchmark_start_timestamp: str,
    benchmark_end_timestamp: str,
    msg: str,
    mae: float,
    rmse: float,
    r2: float,
    chunksize: int = 0,
    resource_stats: dict[str, Any] | None = None,
) -> dict[str, Any]:

    if resource_stats is None:
        resource_stats = ResourceMonitor.as_dict()

    return {
            # operations:
            "stage": QUERIES,
            # Type of operation
            "operation": operation,
            # Test Number
            "test_number": ntest, 
            #Slice name
            "slice_name": slice_name,
            # Slice path
            "parquet_path": str(parquet_folder),
            # Performance
            "total_files": total_files,
            "total_rows_in_dataset": total_rows_in_dataset,
            "total_rows_used_ml": total_rows_used_ml,
            "execution_time_sec": execution_time,
            "throughput_total_rows_sec": throughput,
            "throughput_rows_used_ml_sec": throughput_ml,
            "execution_time_sec_train": execution_time_train,
            "execution_time_sec_pred": execution_time_pred,
            "threads": threads,
            "chunksize": chunksize,
            "parquet_files_size_mib": compute_dataset_size_mib(parquet_folder),
            # Timestamps
            "benchmark_start_timestamp": benchmark_start_timestamp,
            "benchmark_end_timestamp": benchmark_end_timestamp,
            # Resources
            "avg_rss_mib": resource_stats["avg_rss_mib"],
            "peak_rss_mib": resource_stats["peak_rss_mib"],
            "memory_percent_of_total": resource_stats["memory_percent_of_total"],
            "avg_process_cpu_percent": resource_stats["avg_process_cpu_percent"],
            "peak_process_cpu_percent": resource_stats["peak_process_cpu_percent"],
            "avg_system_cpu_per_core_percent": resource_stats["avg_system_cpu_per_core_percent"],
            "peak_system_cpu_per_core_percent": resource_stats["peak_system_cpu_per_core_percent"],
            "read_mib": resource_stats["read_mib"],
            "write_mib": resource_stats["write_mib"], 
            # Quality metrics
            "mae": mae,
            "rmse": rmse,
            "r2": r2,
            # Message
            "msg":  msg
    }

In [46]:
# Benchmark machine learning steps using linear regression to predict tip amount
# Parameters:
# slice_name: key used to resolve the parquet dataset path from PARQUET_PATHS
# ntest: test number
# threads: number of DuckDB execution threads
# logger: Logger instance for reporting progress and errors
# test_size: proportion of the dataset to allocate to the test set
# random_state: seed used to ensure reproducible train/test splits and model behaviour
# Returns:
# benchmark results including slice name, parquet path, total files, total rows in dataset, execution time,
# execution time training, execution time prediction, throughput, threads, timestamps, resource statistics, a message and
# quality metrics
def benchmark_predict_tip_amount(
    slice_name: str | Path, 
    ntest: int,
    threads: int = 0, 
    logger: logging.Logger | None = None, 
    test_size: float = 0.2, 
    random_state: int = 42
) -> dict[str, Any]:

    if logger is None:
        logger = setup_logger()

    timer = Timer()
    monitor = ResourceMonitor()
    benchmark_start_timestamp = Timer.get_timestamp()
    timer.start("total")
    monitor.start()

    MSG_ERROR = "Machine learning algorithm completed with errors, check pipeline.log" 
    parquet_path = PARQUET_PATHS[slice_name]
    parquet_folder = PARQUET_FOLDERS[slice_name]

    total_files = count_parquet_files(parquet_path = parquet_path, use_pattern = True)
    
    if total_files == 0:
        msg = f"No parquet file(s) were found for path: {parquet_folder} OR {parquet_folder} does not exist"
        log_error_msg(logger, msg)
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        return build_return_dict_ml(slice_name = slice_name, parquet_path = parquet_folder, 
                                    operation = IN_MEMORY, ntest = ntest, total_files = total_files, total_rows_in_dataset = 0,
                                    execution_time = execution_time, execution_time_train = 0.0, execution_time_pred = 0.0, throughput = 0.0,
                                    total_rows_used_ml = 0, throughput_ml = 0.0,
                                    threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                    benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), msg = MSG_ERROR, mae = 0.0,
                                    rmse = 0.0, r2 = 0.0, resource_stats = resource_stats)

    connection = None
    
    try:
        connection = duckdb.connect()
        if threads and threads > 0:
            connection.execute(f"PRAGMA threads={threads}")
    except duckdb.Error as e:
        msg = f"Failed to initialize DuckDB connection (threads = {threads}): {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        if connection is not None:
            connection.close()
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        return build_return_dict_ml(slice_name = slice_name, parquet_path = parquet_folder, 
                                    operation = IN_MEMORY, ntest = ntest, total_files = total_files, total_rows_in_dataset = 0,
                                    execution_time = execution_time, execution_time_train = 0.0, execution_time_pred = 0.0, throughput = 0.0,
                                    total_rows_used_ml = 0, throughput_ml = 0.0,
                                    threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                    benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), msg = MSG_ERROR, mae = 0.0,
                                    rmse = 0.0, r2 = 0.0, resource_stats = resource_stats)
    try:
        query = f"""
                 SELECT pickup_datetime, dropoff_datetime, trip_distance, pickup_location_id, 
                        dropoff_location_id, payment_type_id, fare_amount, tip_amount, 
                        year, month FROM read_parquet ('{parquet_path}')
                 """
        df_result = connection.execute(query).fetchdf()
        
    except Exception as e:
        msg = f"Query could not be executed due to the following error: {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        return build_return_dict_ml(slice_name = slice_name, parquet_path = parquet_folder, 
                                    operation = IN_MEMORY, ntest = ntest, total_files = total_files, total_rows_in_dataset = 0,
                                    execution_time = execution_time, execution_time_train = 0.0, execution_time_pred = 0.0, throughput = 0.0,
                                    total_rows_used_ml = 0, throughput_ml = 0.0,
                                    threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                    benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), msg = MSG_ERROR, mae = 0.0,
                                    rmse = 0.0, r2 = 0.0, resource_stats = resource_stats)
          
    finally:
        connection.close()

    if df_result.empty:
        msg = f"No rows found in parquet path: {parquet_path}"
        log_error_msg(logger, msg)
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        return build_return_dict_ml(slice_name = slice_name, parquet_path = parquet_folder, 
                                    operation = IN_MEMORY, ntest = ntest, total_files = total_files, total_rows_in_dataset = 0,
                                    execution_time = execution_time, execution_time_train = 0.0, execution_time_pred = 0.0, throughput = 0.0,
                                    total_rows_used_ml = 0, throughput_ml = 0.0,
                                    threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                    benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), msg = MSG_ERROR, mae = 0.0,
                                    rmse = 0.0, r2 = 0.0, resource_stats = resource_stats)
        
    df_result = transform_ml_features(df_result)
    total_rows = len(df_result)
    target = "tip_amount"
    features = [c for c in df_result.columns if c != target]
    df_result = remove_negative_monetary_value_and_outliers_for_ml(df_result)
    df_result = df_result.dropna(subset = features + [target])
    total_rows_used_ml = len(df_result)

    if total_rows_used_ml == 0:
        log_error_msg(logger, "No valid rows remain after preprocessing")
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time = timer.duration("total")
        throughput = total_rows / execution_time if execution_time > 0 else 0
        return build_return_dict_ml(slice_name = slice_name, parquet_path = parquet_folder, 
                                    operation = IN_MEMORY, ntest = ntest, total_files = total_files, total_rows_in_dataset = total_rows,
                                    execution_time = execution_time, execution_time_train = 0.0, execution_time_pred = 0.0, throughput = throughput,
                                    total_rows_used_ml = total_rows_used_ml, throughput_ml = 0.0,
                                    threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                    benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), msg = MSG_ERROR, mae = 0.0,
                                    rmse = 0.0, r2 = 0.0, resource_stats = resource_stats)
    
    X = df_result[features]
    y = df_result[target]

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size= test_size, random_state = random_state 
    )
    # Scaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train benchmark
    timer.start("train")
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    timer.stop("train")

    # Prediction benchmark
    timer.start("pred")
    y_pred = model.predict(X_test_scaled)
    timer.stop("pred")
    
    timer.stop("total")
    resource_stats = monitor.stop()
    benchmark_end_timestamp = Timer.get_timestamp()
    
    # Performance metrics
    execution_time = timer.duration("total")
    execution_time_train = timer.duration("train")
    execution_time_pred = timer.duration("pred")
    throughput = total_rows / execution_time if execution_time > 0 else 0.0
    throughput_ml = total_rows_used_ml / execution_time if execution_time > 0 else 0.0

    # Quality metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    print(f"Pandas and Scikit-learn run the training algorithm in {execution_time_train:.2f} seconds.")
    print(f"Pandas and Scikit-learn run the prediction algorithm in {execution_time_pred:.2f} seconds.")    
    print(f"Pandas and Scikit-learn run the machine learning algorithm in {execution_time:.2f} seconds. Processed {total_rows} row(s)")
    print(f"Throughput: {throughput:.2f} rows/sec")

    return build_return_dict_ml(slice_name = slice_name, parquet_path = parquet_folder, 
                                operation = IN_MEMORY, ntest = ntest, total_files = total_files, total_rows_in_dataset = total_rows,
                                execution_time = execution_time, execution_time_train = execution_time_train, execution_time_pred = execution_time_pred, 
                                total_rows_used_ml = total_rows_used_ml, throughput_ml = throughput_ml,
                                throughput = throughput, threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), msg = "Machine learning algorithm completed", mae = mae,
                                rmse = rmse, r2 = r2, resource_stats = resource_stats)  

In [138]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
result = benchmark_predict_tip_amount(slice_name = "1st-slice", logger = logger, ntest = 1, threads = 3)
log_info_msg(logger, result["msg"])

Pandas and Scikit-learn run the training algorithm in 3.77 seconds.
Pandas and Scikit-learn run the prediction algorithm in 0.12 seconds.
Pandas and Scikit-learn run the machine learning algorithm in 38.04 seconds. Processed 3842011 row(s)
Throughput: 101004.11 rows/sec


In [139]:
print(result)

{'stage': 'queries', 'operation': 'standard', 'test_number': 1, 'slice_name': '1st-slice', 'parquet_path': '../data/parquet/1st-slice_compacted/**/*.parquet', 'total_files': 1, 'total_rows_in_dataset': 3842011, 'total_rows_used_ml': 3594852, 'execution_time_sec': 38.03816439299953, 'throughput_total_rows_sec': 101004.11156294061, 'throughput_rows_used_ml_sec': 94506.45312058195, 'execution_time_sec_train': 3.770622992000426, 'execution_time_sec_pred': 0.12115921300028276, 'threads': 3, 'chunksize': 0, 'parquet_files_size_mib': 79.27833938598633, 'benchmark_start_timestamp': '2026-04-01T00:10:20.132734+01:00', 'benchmark_end_timestamp': '2026-04-01T00:10:58.256308+01:00', 'avg_rss_mib': 1583.4965672348485, 'peak_rss_mib': 2684.46875, 'memory_percent_of_total': 34.215799958078144, 'avg_process_cpu_percent': 191.6969696969697, 'peak_process_cpu_percent': 235.6, 'avg_system_cpu_per_core_percent': [40.77272727272728, 30.53939393939394, 42.96969696969697, 42.06969696969698], 'peak_system_cpu

##### Training, incremental learning

In [137]:
# Benchmark machine learning steps using incremental learning with SGDRRegressor
# Parameters:
# slice_name: key used to resolve the parquet dataset path from PARQUET_PATHS
# ntest: test number
# approx_rows_per_chunk: approximate number of rows to process per chunk during incremental learning
# threads: number of DuckDB execution threads
# logger: Logger instance for reporting progress and errors
# test_size: proportion of the dataset to allocate to the test set
# random_state: seed used to ensure reproducible train/test splits and model behaviour
# Returns:
# benchmark results including slice name, parquet path, total files, total rows in dataset, execution time,
# execution time training, execution time prediction, throughput, threads, timestamps, resource statistics, a message and
# quality metrics
def benchmark_predict_tip_amount_incremental_learning(
    slice_name: str | Path, 
    ntest: int,
    approx_rows_per_chunk: int = 100_000, 
    threads: int = 0, 
    logger: logging.Logger | None = None, 
    test_size: float = 0.2, 
    random_state: int = 42
) -> dict[str, Any]:

    if logger is None:
        logger = setup_logger()

    timer = Timer()
    monitor = ResourceMonitor()
    benchmark_start_timestamp = Timer.get_timestamp()
    timer.start("total")
    timer.start("train")
    monitor.start()

    MSG_ERROR = "Machine learning algorithm completed with errors, check pipeline.log" 
    
    parquet_path = PARQUET_PATHS[slice_name]
    parquet_folder = PARQUET_FOLDERS[slice_name]

    total_files = count_parquet_files(parquet_path,  use_pattern = True)
    
    if total_files == 0:
        msg = f"No parquet file(s) were found for path: {parquet_folder} OR {parquet_folder} does not exist"
        log_error_msg(logger, msg)
        timer.stop("train")
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time_train = timer.duration("train")
        execution_time = timer.duration("total")
        return build_return_dict_ml(slice_name = slice_name, parquet_path = parquet_folder, 
                                    operation = CHUNKING, ntest = ntest, total_files = total_files, total_rows_in_dataset = 0,
                                    execution_time = execution_time, execution_time_train = execution_time_train, execution_time_pred = 0.0, throughput = 0.0,
                                    total_rows_used_ml = 0, throughput_ml = 0.0,
                                    threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                    benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), msg = MSG_ERROR, mae = 0.0,
                                    chunksize = max(1, approx_rows_per_chunk // 2048), rmse = 0.0, r2 = 0.0, 
                                    resource_stats = resource_stats)

    connection = None
    
    try:
        connection = duckdb.connect()
        if threads and threads > 0:
            connection.execute(f"PRAGMA threads={threads}")
    except duckdb.Error as e:
        msg = f"Failed to initialize DuckDB connection (threads = {threads}): {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        if connection is not None:
            connection.close()
        timer.stop("train")
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time_train = timer.duration("train")
        execution_time = timer.duration("total")
        return build_return_dict_ml(slice_name = slice_name, parquet_path = parquet_folder, 
                                    operation = CHUNKING, ntest = ntest, total_files = total_files, total_rows_in_dataset = 0,
                                    execution_time = execution_time, execution_time_train = execution_time_train, execution_time_pred = 0.0, throughput = 0.0,
                                    total_rows_used_ml = 0, throughput_ml = 0.0,
                                    threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                    benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), msg = MSG_ERROR, mae = 0.0,
                                    chunksize = max(1, approx_rows_per_chunk // 2048), rmse = 0.0, r2 = 0.0, 
                                    resource_stats = resource_stats)

    total_rows = 0
    total_rows_used_ml = 0
    X_test_scaled_parts = []
    y_test_parts = []

    try:
        
        query = f"""
                 SELECT pickup_datetime, dropoff_datetime, trip_distance, pickup_location_id, 
                        dropoff_location_id, payment_type_id, fare_amount, tip_amount,
                        year, month FROM read_parquet ('{parquet_path}')
                 """
        vector_multiple = max(1, approx_rows_per_chunk // 2048)
        df_result = connection.execute(query)
        model = SGDRegressor(random_state = random_state)
        scaler = StandardScaler()
    
        target = "tip_amount"

        while True:
            
            chunk = df_result.fetch_df_chunk(vector_multiple)
            
            if chunk is None or chunk.empty:
                break

            total_rows += len(chunk) 

            chunk = transform_ml_features(chunk)
            features = [c for c in chunk.columns if c != target]
            chunk = remove_negative_monetary_value_and_outliers_for_ml(chunk)
            chunk = chunk.dropna(subset = features + [target])
            
            if chunk.empty:
                continue

            total_rows_used_ml += len(chunk)
            
            X = chunk[features]
            y = chunk[target]    

            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = test_size, random_state = random_state)

            if len(X_train) == 0:
                continue
            
            scaler.partial_fit(X_train)

            X_train_scaled = scaler.transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            model.partial_fit(X_train_scaled, y_train)

            if len(X_test_scaled) > 0:
                X_test_scaled_parts.append(X_test_scaled)
                y_test_parts.append(y_test.to_numpy())
                
    except Exception as e:
        msg = f"Training step failed: {type(e).__name__}: {e}"
        log_error_msg(logger, msg)
        timer.stop("train")
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time_train = timer.duration("train")
        execution_time = timer.duration("total")
        throughput = total_rows / execution_time if execution_time > 0 else 0.0
        throughput_ml = total_rows_used_ml / execution_time if execution_time > 0 else 0.0
        return build_return_dict_ml(slice_name = slice_name, parquet_path = parquet_folder, 
                                    operation = CHUNKING, ntest = ntest, total_files = total_files, total_rows_in_dataset = 0,
                                    execution_time = execution_time, execution_time_train = execution_time_train, execution_time_pred = 0.0, throughput = throughput,
                                    total_rows_used_ml = total_rows_used_ml, throughput_ml = throughput_ml,
                                    threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                    benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), msg = MSG_ERROR, mae = 0.0,
                                    chunksize = max(1, approx_rows_per_chunk // 2048), rmse = 0.0, r2 = 0.0, 
                                    resource_stats = resource_stats)
            
    finally:
        connection.close()

    timer.stop("train")

    # Prediction benchmark
    if not X_test_scaled_parts or not y_test_parts:
        log_error_msg(logger, "Prediction step could be not performed: no test data collected")
        timer.stop("total")
        resource_stats = monitor.stop()
        benchmark_end_timestamp = Timer.get_timestamp()
        execution_time_train = timer.duration("train")
        execution_time = timer.duration("total")
        throughput = total_rows / execution_time if execution_time > 0 else 0.0
        throughput_ml = total_rows_used_ml / execution_time if execution_time > 0 else 0.0
        return build_return_dict_ml(slice_name = slice_name, parquet_path = parquet_folder, 
                                    operation = CHUNKING, ntest = ntest, total_files = total_files, total_rows_in_dataset = 0,
                                    execution_time = execution_time, execution_time_train = execution_time_train, execution_time_pred = 0.0, throughput = throughput,
                                    total_rows_used_ml = total_rows_used_ml, throughput_ml = throughput_ml,
                                    threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                    benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), msg = MSG_ERROR, mae = 0.0,
                                    chunksize = max(1, approx_rows_per_chunk // 2048), rmse = 0.0, r2 = 0.0, 
                                    resource_stats = resource_stats)
         
    timer.start("pred")
    X_test_final = np.vstack(X_test_scaled_parts)
    y_test_final = np.concatenate(y_test_parts)
    y_pred = model.predict(X_test_final)
    timer.stop("pred")
    timer.stop("total")
    resource_stats = monitor.stop()
    benchmark_end_timestamp = Timer.get_timestamp()
    
    # Performance metrics
    execution_time_train = timer.duration("train")
    execution_time_pred = timer.duration("pred")
    execution_time = timer.duration("total")  
    throughput = total_rows / execution_time if execution_time > 0 else 0.0
    throughput_ml = total_rows_used_ml / execution_time if execution_time > 0 else 0.0
    
    # Quality metrics
    mae = mean_absolute_error(y_test_final, y_pred)
    rmse = mean_squared_error(y_test_final, y_pred) ** 0.5
    r2 = r2_score(y_test_final, y_pred)
    
    print(f"Pandas and Scikit-learn run the training algorithm in {execution_time_train:.2f} seconds.")
    print(f"Pandas and Scikit-learn run the prediction algorithm in {execution_time_pred:.2f} seconds.")    
    print(f"Pandas and Scikit-learn run the machine learning algorithm in {execution_time:.2f} seconds. Processed {total_rows} row(s)")
    print(f"Throughput: {throughput:.2f} rows/sec")
    
    return build_return_dict_ml(slice_name = slice_name, parquet_path = parquet_folder, 
                                operation = CHUNKING, ntest = ntest, total_files = total_files, total_rows_in_dataset = total_rows,
                                execution_time = execution_time, execution_time_train = execution_time_train, execution_time_pred = execution_time_pred, 
                                throughput = throughput, total_rows_used_ml = total_rows_used_ml, throughput_ml = throughput_ml,
                                threads = threads, benchmark_start_timestamp = benchmark_start_timestamp.isoformat(), 
                                benchmark_end_timestamp = benchmark_end_timestamp.isoformat(), msg = "Machine learning algorithm completed", mae = mae,
                                chunksize = max(1, approx_rows_per_chunk // 2048), rmse = rmse, r2 = r2, 
                                resource_stats = resource_stats)

In [150]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
result = benchmark_predict_tip_amount_incremental_learning(slice_name = "2nd-slice", logger = logger, ntest = 1, threads = 3)
log_info_msg(logger, result["msg"])

Pandas and Scikit-learn run the training algorithm in 308.17 seconds.
Pandas and Scikit-learn run the prediction algorithm in 1.50 seconds.
Pandas and Scikit-learn run the machine learning algorithm in 309.67 seconds. Processed 12621541 row(s)
Throughput: 40758.53 rows/sec


In [151]:
print(result)

{'stage': 'queries', 'operation': 'chunking', 'test_number': 1, 'slice_name': '2nd-slice', 'parquet_path': '../data/parquet/2nd-slice_compacted/**/*.parquet', 'total_files': 3, 'total_rows_in_dataset': 12621541, 'total_rows_used_ml': 11773494, 'execution_time_sec': 309.66626469900075, 'throughput_total_rows_sec': 40758.52761122781, 'throughput_rows_used_ml_sec': 38019.94386260956, 'execution_time_sec_train': 308.1672882040002, 'execution_time_sec_pred': 1.4989555850006582, 'threads': 3, 'chunksize': 48, 'parquet_files_size_mib': 260.6450300216675, 'benchmark_start_timestamp': '2026-04-01T00:46:27.398644+01:00', 'benchmark_end_timestamp': '2026-04-01T00:51:37.751022+01:00', 'avg_rss_mib': 1009.9478711790393, 'peak_rss_mib': 1374.01171875, 'memory_percent_of_total': 17.512928809026043, 'avg_process_cpu_percent': 149.13842794759822, 'peak_process_cpu_percent': 196.1, 'avg_system_cpu_per_core_percent': [31.73187772925765, 30.106550218340605, 30.75676855895198, 30.107423580786048], 'peak_sy

In [152]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
result = benchmark_predict_tip_amount_incremental_learning(slice_name = "3rd-slice", logger = logger, ntest = 1, threads = 3)
log_info_msg(logger, result["msg"])

Pandas and Scikit-learn run the training algorithm in 1414.10 seconds.
Pandas and Scikit-learn run the prediction algorithm in 7.16 seconds.
Pandas and Scikit-learn run the machine learning algorithm in 1421.26 seconds. Processed 52084283 row(s)
Throughput: 36646.53 rows/sec


In [153]:
print(result)

{'stage': 'queries', 'operation': 'chunking', 'test_number': 1, 'slice_name': '3rd-slice', 'parquet_path': '../data/parquet/3rd-slice_compacted/**/*.parquet', 'total_files': 14, 'total_rows_in_dataset': 52084283, 'total_rows_used_ml': 50036568, 'execution_time_sec': 1421.2609826549997, 'throughput_total_rows_sec': 36646.52983205342, 'throughput_rows_used_ml_sec': 35205.75644490622, 'execution_time_sec_train': 1414.0962831849993, 'execution_time_sec_pred': 7.16467838599965, 'threads': 3, 'chunksize': 48, 'parquet_files_size_mib': 1072.2892599105835, 'benchmark_start_timestamp': '2026-04-01T00:52:35.601722+01:00', 'benchmark_end_timestamp': '2026-04-01T01:16:17.191370+01:00', 'avg_rss_mib': 1494.097831320028, 'peak_rss_mib': 3177.9296875, 'memory_percent_of_total': 40.505372419901626, 'avg_process_cpu_percent': 153.15088702147528, 'peak_process_cpu_percent': 201.4, 'avg_system_cpu_per_core_percent': [31.223529411764748, 29.813352007469607, 29.620448179271698, 30.490382819794572], 'peak_s

In [154]:
if logger is None:
    logger = setup_logger()
else:
    if logger.handlers:
        logger.handlers.clear()
    logger = setup_logger()
    
result = benchmark_predict_tip_amount_incremental_learning(slice_name = "4th-slice", logger = logger, ntest = 1, threads = 3)
log_info_msg(logger, result["msg"])

Pandas and Scikit-learn run the training algorithm in 3116.44 seconds.
Pandas and Scikit-learn run the prediction algorithm in 26.37 seconds.
Pandas and Scikit-learn run the machine learning algorithm in 3142.81 seconds. Processed 107144513 row(s)
Throughput: 34091.93 rows/sec


In [155]:
print(result)

{'stage': 'queries', 'operation': 'chunking', 'test_number': 1, 'slice_name': '4th-slice', 'parquet_path': '../data/parquet/4th-slice_compacted/**/*.parquet', 'total_files': 31, 'total_rows_in_dataset': 107144513, 'total_rows_used_ml': 104460378, 'execution_time_sec': 3142.8119546379985, 'throughput_total_rows_sec': 34091.926130636515, 'throughput_rows_used_ml_sec': 33237.87089642535, 'execution_time_sec_train': 3116.440261488, 'execution_time_sec_pred': 26.371674600000915, 'threads': 3, 'chunksize': 48, 'parquet_files_size_mib': 2199.0808296203613, 'benchmark_start_timestamp': '2026-04-01T01:19:10.076589+01:00', 'benchmark_end_timestamp': '2026-04-01T02:11:33.555891+01:00', 'avg_rss_mib': 1918.7673033134422, 'peak_rss_mib': 5844.06640625, 'memory_percent_of_total': 74.48751530371686, 'avg_process_cpu_percent': 154.88714405360147, 'peak_process_cpu_percent': 285.1, 'avg_system_cpu_per_core_percent': [30.492001675041923, 28.945058626465716, 29.105527638190893, 30.454480737018432], 'peak

In [114]:
import duckdb

con = duckdb.connect()
schema = con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('../data/parquet/4th-slice_compacted/year=2023/month=4/compacted_data_20260325_202210_c077e170641f4ca1a1c4827cbf3006db.parquet')
""").fetchdf()
con.close()
print(schema)

               column_name   column_type null   key default extra
0                vendor_id      UTINYINT  YES  None    None  None
1          pickup_datetime  TIMESTAMP_NS  YES  None    None  None
2         dropoff_datetime  TIMESTAMP_NS  YES  None    None  None
3          passenger_count      UTINYINT  YES  None    None  None
4            trip_distance         FLOAT  YES  None    None  None
5             rate_code_id      UTINYINT  YES  None    None  None
6   store_and_forward_flag       VARCHAR  YES  None    None  None
7       pickup_location_id      SMALLINT  YES  None    None  None
8      dropoff_location_id      SMALLINT  YES  None    None  None
9          payment_type_id      UTINYINT  YES  None    None  None
10             fare_amount         FLOAT  YES  None    None  None
11            extra_charge         FLOAT  YES  None    None  None
12        metered_rate_tax         FLOAT  YES  None    None  None
13              tip_amount         FLOAT  YES  None    None  None
14        